# XGBoost Best-Iteration, Model-Capacity and GA Stability Analysis

This notebook investigates XGBoost model complexity and training duration using the GA-selected Mordred descriptors. It compares shallow, standard and deeper tree configurations, identifies validation-selected boosting iterations, and evaluates full Mordred, mRMR and GA feature sets under comparable training settings. The notebook also examines multi-seed GA stability and builds the final seed-84 XGBoost model before evaluation on the held-out scaffold test set.

In [ ]:
# Import required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from xgboost import XGBClassifier

from sklearn.utils.class_weight import compute_sample_weight

from sklearn.metrics import (
    matthews_corrcoef,
    balanced_accuracy_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Set random seed
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

print("Libraries imported successfully.")
print("Random seed:", RANDOM_SEED)

In [ ]:
# Load training and validation data

X_train = pd.read_pickle(
    "mordred_training_features_filtered.pkl"
)

X_valid = pd.read_pickle(
    "mordred_validation_features_filtered.pkl"
)

y_train = pd.read_pickle(
    "mordred_training_labels.pkl"
)

y_valid = pd.read_pickle(
    "mordred_validation_labels.pkl"
)

# Load the 316 descriptors selected by the GA

ga_features_data = pd.read_csv(
    "full_ga_results/full_ga_best_features.csv"
)

# Remove an unwanted saved index column if present
ga_features_data = ga_features_data.loc[
    :,
    ~ga_features_data.columns.str.startswith("Unnamed")
]

# Use the descriptor column
if "descriptor" in ga_features_data.columns:
    ga_features = ga_features_data["descriptor"].tolist()
else:
    ga_features = ga_features_data.iloc[:, 0].tolist()

# Check whether all GA descriptors are available
missing_features = [
    feature for feature in ga_features
    if feature not in X_train.columns
    or feature not in X_valid.columns
]

print("Training features shape:", X_train.shape)
print("Validation features shape:", X_valid.shape)
print("Training labels:", len(y_train))
print("Validation labels:", len(y_valid))
print("GA-selected descriptors:", len(ga_features))
print("Missing GA descriptors:", len(missing_features))

if len(missing_features) == 0:
    print("All required data loaded successfully.")
else:
    print("Missing descriptors:", missing_features[:10])

In [ ]:
# Find the boosting iteration with the lowest validation loss

# Calculate balanced training weights
train_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

# Current GA model with more boosting rounds
ga_best_iteration_model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=3000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.10,
    reg_lambda=1.00,
    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=200,
    random_state=42,
    n_jobs=-1
)

# Train while monitoring training and validation loss
ga_best_iteration_model.fit(
    X_train[ga_features],
    y_train,
    sample_weight=train_weights,
    eval_set=[
        (X_train[ga_features], y_train),
        (X_valid[ga_features], y_valid)
    ],
    verbose=False
)

# Get the learning history
history = ga_best_iteration_model.evals_result()

training_loss = history["validation_0"]["logloss"]
validation_loss = history["validation_1"]["logloss"]

best_iteration = ga_best_iteration_model.best_iteration
rounds_completed = len(validation_loss)
last_iteration = rounds_completed - 1

print("Maximum boosting rounds:", 3000)
print("Rounds completed:", rounds_completed)
print("Best iteration:", best_iteration)
print(
    "Minimum validation log-loss:",
    round(validation_loss[best_iteration], 6)
)
print(
    "Validation log-loss at last completed iteration:",
    round(validation_loss[-1], 6)
)

# Predictions using the best iteration
train_prob_best = ga_best_iteration_model.predict_proba(
    X_train[ga_features],
    iteration_range=(0, best_iteration + 1)
)[:, 1]

valid_prob_best = ga_best_iteration_model.predict_proba(
    X_valid[ga_features],
    iteration_range=(0, best_iteration + 1)
)[:, 1]

train_pred_best = (train_prob_best >= 0.50).astype(int)
valid_pred_best = (valid_prob_best >= 0.50).astype(int)

# Predictions using the last completed iteration
valid_prob_last = ga_best_iteration_model.predict_proba(
    X_valid[ga_features],
    iteration_range=(0, rounds_completed)
)[:, 1]

valid_pred_last = (valid_prob_last >= 0.50).astype(int)

# Calculate performance
train_mcc_best = matthews_corrcoef(
    y_train,
    train_pred_best
)

valid_mcc_best = matthews_corrcoef(
    y_valid,
    valid_pred_best
)

valid_mcc_last = matthews_corrcoef(
    y_valid,
    valid_pred_last
)

train_ba_best = balanced_accuracy_score(
    y_train,
    train_pred_best
)

valid_ba_best = balanced_accuracy_score(
    y_valid,
    valid_pred_best
)

valid_ba_last = balanced_accuracy_score(
    y_valid,
    valid_pred_last
)

# Create result table
best_iteration_results = pd.DataFrame([{
    "rounds_completed": rounds_completed,
    "best_iteration": best_iteration,
    "minimum_validation_logloss":
        validation_loss[best_iteration],
    "last_validation_logloss":
        validation_loss[-1],
    "training_mcc_best_iteration":
        train_mcc_best,
    "validation_mcc_best_iteration":
        valid_mcc_best,
    "validation_mcc_last_iteration":
        valid_mcc_last,
    "training_validation_mcc_gap":
        train_mcc_best - valid_mcc_best,
    "training_balanced_accuracy":
        train_ba_best,
    "validation_balanced_accuracy_best":
        valid_ba_best,
    "validation_balanced_accuracy_last":
        valid_ba_last
}])

display(best_iteration_results)

# Save results
best_iteration_results.to_csv(
    "ga_best_iteration_results.csv",
    index=False
)

# Plot learning curves
plt.figure(figsize=(9, 5))

plt.plot(
    range(rounds_completed),
    training_loss,
    label="Training log-loss"
)

plt.plot(
    range(rounds_completed),
    validation_loss,
    label="Validation log-loss"
)

plt.axvline(
    best_iteration,
    linestyle="--",
    label=f"Best iteration: {best_iteration}"
)

plt.xlabel("Boosting iteration")
plt.ylabel("Log-loss")
plt.title("GA Model Training and Validation Log-loss")
plt.legend()
plt.tight_layout()

plt.savefig(
    "ga_best_iteration_learning_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print("ga_best_iteration_results.csv")
print("ga_best_iteration_learning_curve.png")

In [ ]:
# Save and review the best-iteration model

ga_best_iteration_model.save_model(
    "ga_best_iteration_model.json"
)

print("Best-iteration results:\n")

for column, value in best_iteration_results.iloc[0].items():
    if isinstance(value, float):
        print(f"{column}: {value:.6f}")
    else:
        print(f"{column}: {value}")

print("\nSelected boosting iteration:", best_iteration)
print("Number of trees used:", best_iteration + 1)

print("\nSaved:")
print("ga_best_iteration_model.json")

In [ ]:
# Train a very shallow model

shallow_model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=3000,
    learning_rate=0.05,
    max_depth=2,
    min_child_weight=3,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.10,
    reg_lambda=1.00,
    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=200,
    random_state=42,
    n_jobs=-1
)

shallow_model.fit(
    X_train[ga_features],
    y_train,
    sample_weight=train_weights,
    eval_set=[
        (X_train[ga_features], y_train),
        (X_valid[ga_features], y_valid)
    ],
    verbose=False
)

# Get training history
shallow_history = shallow_model.evals_result()

shallow_training_loss = shallow_history[
    "validation_0"
]["logloss"]

shallow_validation_loss = shallow_history[
    "validation_1"
]["logloss"]

shallow_best_iteration = shallow_model.best_iteration
shallow_rounds_completed = len(shallow_validation_loss)

# Predictions using the best iteration
shallow_train_probability = shallow_model.predict_proba(
    X_train[ga_features],
    iteration_range=(0, shallow_best_iteration + 1)
)[:, 1]

shallow_valid_probability = shallow_model.predict_proba(
    X_valid[ga_features],
    iteration_range=(0, shallow_best_iteration + 1)
)[:, 1]

shallow_train_prediction = (
    shallow_train_probability >= 0.50
).astype(int)

shallow_valid_prediction = (
    shallow_valid_probability >= 0.50
).astype(int)

# Calculate performance
shallow_train_mcc = matthews_corrcoef(
    y_train,
    shallow_train_prediction
)

shallow_valid_mcc = matthews_corrcoef(
    y_valid,
    shallow_valid_prediction
)

shallow_train_ba = balanced_accuracy_score(
    y_train,
    shallow_train_prediction
)

shallow_valid_ba = balanced_accuracy_score(
    y_valid,
    shallow_valid_prediction
)

# Save results
shallow_results = pd.DataFrame([{
    "model": "Very shallow",
    "max_depth": 2,
    "rounds_completed": shallow_rounds_completed,
    "best_iteration": shallow_best_iteration,
    "minimum_validation_logloss":
        shallow_validation_loss[shallow_best_iteration],
    "training_mcc": shallow_train_mcc,
    "validation_mcc": shallow_valid_mcc,
    "training_validation_mcc_gap":
        shallow_train_mcc - shallow_valid_mcc,
    "training_balanced_accuracy":
        shallow_train_ba,
    "validation_balanced_accuracy":
        shallow_valid_ba
}])

shallow_results.to_csv(
    "ga_shallow_best_iteration_results.csv",
    index=False
)

shallow_model.save_model(
    "ga_shallow_best_iteration_model.json"
)

display(shallow_results)

# Plot learning curves
plt.figure(figsize=(9, 5))

plt.plot(
    range(shallow_rounds_completed),
    shallow_training_loss,
    label="Training log-loss"
)

plt.plot(
    range(shallow_rounds_completed),
    shallow_validation_loss,
    label="Validation log-loss"
)

plt.axvline(
    shallow_best_iteration,
    linestyle="--",
    label=f"Best iteration: {shallow_best_iteration}"
)

plt.xlabel("Boosting iteration")
plt.ylabel("Log-loss")
plt.title("Very Shallow GA Model Training and Validation Log-loss")
plt.legend()
plt.tight_layout()

plt.savefig(
    "ga_shallow_best_iteration_learning_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print("ga_shallow_best_iteration_results.csv")
print("ga_shallow_best_iteration_model.json")
print("ga_shallow_best_iteration_learning_curve.png")

In [ ]:
# Extend the very shallow model to 5000 boosting rounds

shallow_extended_model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=5000,
    learning_rate=0.05,
    max_depth=2,
    min_child_weight=3,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.10,
    reg_lambda=1.00,
    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=200,
    random_state=42,
    n_jobs=-1
)

shallow_extended_model.fit(
    X_train[ga_features],
    y_train,
    sample_weight=train_weights,
    eval_set=[
        (X_train[ga_features], y_train),
        (X_valid[ga_features], y_valid)
    ],
    verbose=False
)

# Get learning history
shallow_extended_history = (
    shallow_extended_model.evals_result()
)

shallow_extended_train_loss = (
    shallow_extended_history[
        "validation_0"
    ]["logloss"]
)

shallow_extended_valid_loss = (
    shallow_extended_history[
        "validation_1"
    ]["logloss"]
)

shallow_extended_best_iteration = (
    shallow_extended_model.best_iteration
)

shallow_extended_rounds = len(
    shallow_extended_valid_loss
)

# Predictions at the best iteration
shallow_extended_train_prob = (
    shallow_extended_model.predict_proba(
        X_train[ga_features],
        iteration_range=(
            0,
            shallow_extended_best_iteration + 1
        )
    )[:, 1]
)

shallow_extended_valid_prob = (
    shallow_extended_model.predict_proba(
        X_valid[ga_features],
        iteration_range=(
            0,
            shallow_extended_best_iteration + 1
        )
    )[:, 1]
)

shallow_extended_train_pred = (
    shallow_extended_train_prob >= 0.50
).astype(int)

shallow_extended_valid_pred = (
    shallow_extended_valid_prob >= 0.50
).astype(int)

# Calculate performance
shallow_extended_train_mcc = (
    matthews_corrcoef(
        y_train,
        shallow_extended_train_pred
    )
)

shallow_extended_valid_mcc = (
    matthews_corrcoef(
        y_valid,
        shallow_extended_valid_pred
    )
)

shallow_extended_train_ba = (
    balanced_accuracy_score(
        y_train,
        shallow_extended_train_pred
    )
)

shallow_extended_valid_ba = (
    balanced_accuracy_score(
        y_valid,
        shallow_extended_valid_pred
    )
)

# Save results
shallow_extended_results = pd.DataFrame([{
    "model": "Very shallow extended",
    "max_depth": 2,
    "rounds_completed":
        shallow_extended_rounds,
    "best_iteration":
        shallow_extended_best_iteration,
    "minimum_validation_logloss":
        shallow_extended_valid_loss[
            shallow_extended_best_iteration
        ],
    "training_mcc":
        shallow_extended_train_mcc,
    "validation_mcc":
        shallow_extended_valid_mcc,
    "training_validation_mcc_gap":
        shallow_extended_train_mcc
        - shallow_extended_valid_mcc,
    "training_balanced_accuracy":
        shallow_extended_train_ba,
    "validation_balanced_accuracy":
        shallow_extended_valid_ba
}])

shallow_extended_results.to_csv(
    "ga_shallow_5000_round_results.csv",
    index=False
)

shallow_extended_model.save_model(
    "ga_shallow_5000_round_model.json"
)

display(shallow_extended_results)

# Plot learning curves
plt.figure(figsize=(9, 5))

plt.plot(
    range(shallow_extended_rounds),
    shallow_extended_train_loss,
    label="Training log-loss"
)

plt.plot(
    range(shallow_extended_rounds),
    shallow_extended_valid_loss,
    label="Validation log-loss"
)

plt.axvline(
    shallow_extended_best_iteration,
    linestyle="--",
    label=(
        f"Best iteration: "
        f"{shallow_extended_best_iteration}"
    )
)

plt.xlabel("Boosting iteration")
plt.ylabel("Log-loss")
plt.title(
    "Very Shallow GA Model: "
    "Training and Validation Log-loss"
)
plt.legend()
plt.tight_layout()

plt.savefig(
    "ga_shallow_5000_round_learning_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print("ga_shallow_5000_round_results.csv")
print("ga_shallow_5000_round_model.json")
print("ga_shallow_5000_round_learning_curve.png")

In [ ]:
# Extend the very shallow model to 10000 rounds

shallow_final_model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=10000,
    learning_rate=0.05,
    max_depth=2,
    min_child_weight=3,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.10,
    reg_lambda=1.00,
    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=300,
    random_state=42,
    n_jobs=-1
)

shallow_final_model.fit(
    X_train[ga_features],
    y_train,
    sample_weight=train_weights,
    eval_set=[
        (X_train[ga_features], y_train),
        (X_valid[ga_features], y_valid)
    ],
    verbose=False
)

# Get the learning history
shallow_final_history = shallow_final_model.evals_result()

shallow_final_train_loss = (
    shallow_final_history["validation_0"]["logloss"]
)

shallow_final_valid_loss = (
    shallow_final_history["validation_1"]["logloss"]
)

shallow_final_best_iteration = (
    shallow_final_model.best_iteration
)

shallow_final_rounds = len(
    shallow_final_valid_loss
)

# Predictions using the best iteration
shallow_final_train_prob = shallow_final_model.predict_proba(
    X_train[ga_features],
    iteration_range=(
        0,
        shallow_final_best_iteration + 1
    )
)[:, 1]

shallow_final_valid_prob = shallow_final_model.predict_proba(
    X_valid[ga_features],
    iteration_range=(
        0,
        shallow_final_best_iteration + 1
    )
)[:, 1]

shallow_final_train_pred = (
    shallow_final_train_prob >= 0.50
).astype(int)

shallow_final_valid_pred = (
    shallow_final_valid_prob >= 0.50
).astype(int)

# Calculate performance
shallow_final_train_mcc = matthews_corrcoef(
    y_train,
    shallow_final_train_pred
)

shallow_final_valid_mcc = matthews_corrcoef(
    y_valid,
    shallow_final_valid_pred
)

shallow_final_train_ba = balanced_accuracy_score(
    y_train,
    shallow_final_train_pred
)

shallow_final_valid_ba = balanced_accuracy_score(
    y_valid,
    shallow_final_valid_pred
)

# Save results
shallow_final_results = pd.DataFrame([{
    "model": "Very shallow final",
    "max_depth": 2,
    "rounds_completed": shallow_final_rounds,
    "best_iteration": shallow_final_best_iteration,
    "minimum_validation_logloss":
        shallow_final_valid_loss[
            shallow_final_best_iteration
        ],
    "training_mcc": shallow_final_train_mcc,
    "validation_mcc": shallow_final_valid_mcc,
    "training_validation_mcc_gap":
        shallow_final_train_mcc
        - shallow_final_valid_mcc,
    "training_balanced_accuracy":
        shallow_final_train_ba,
    "validation_balanced_accuracy":
        shallow_final_valid_ba
}])

display(shallow_final_results)

shallow_final_results.to_csv(
    "ga_shallow_10000_round_results.csv",
    index=False
)

shallow_final_model.save_model(
    "ga_shallow_10000_round_model.json"
)

# Plot the learning curves
plt.figure(figsize=(9, 5))

plt.plot(
    range(shallow_final_rounds),
    shallow_final_train_loss,
    label="Training log-loss"
)

plt.plot(
    range(shallow_final_rounds),
    shallow_final_valid_loss,
    label="Validation log-loss"
)

plt.axvline(
    shallow_final_best_iteration,
    linestyle="--",
    label=f"Best iteration: {shallow_final_best_iteration}"
)

plt.xlabel("Boosting iteration")
plt.ylabel("Log-loss")
plt.title(
    "Very Shallow GA Model: "
    "Training and Validation Log-loss"
)
plt.legend()
plt.tight_layout()

plt.savefig(
    "ga_shallow_10000_round_learning_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print("ga_shallow_10000_round_results.csv")
print("ga_shallow_10000_round_model.json")
print("ga_shallow_10000_round_learning_curve.png")

In [ ]:
# Train a high-capacity model

deep_model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=3000,
    learning_rate=0.05,
    max_depth=12,
    min_child_weight=3,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.10,
    reg_lambda=1.00,
    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=200,
    random_state=42,
    n_jobs=-1
)

deep_model.fit(
    X_train[ga_features],
    y_train,
    sample_weight=train_weights,
    eval_set=[
        (X_train[ga_features], y_train),
        (X_valid[ga_features], y_valid)
    ],
    verbose=False
)

# Get the learning history
deep_history = deep_model.evals_result()

deep_training_loss = deep_history[
    "validation_0"
]["logloss"]

deep_validation_loss = deep_history[
    "validation_1"
]["logloss"]

deep_best_iteration = deep_model.best_iteration
deep_rounds_completed = len(deep_validation_loss)

# Predictions using the best iteration
deep_train_probability = deep_model.predict_proba(
    X_train[ga_features],
    iteration_range=(0, deep_best_iteration + 1)
)[:, 1]

deep_valid_probability = deep_model.predict_proba(
    X_valid[ga_features],
    iteration_range=(0, deep_best_iteration + 1)
)[:, 1]

deep_train_prediction = (
    deep_train_probability >= 0.50
).astype(int)

deep_valid_prediction = (
    deep_valid_probability >= 0.50
).astype(int)

# Calculate performance
deep_train_mcc = matthews_corrcoef(
    y_train,
    deep_train_prediction
)

deep_valid_mcc = matthews_corrcoef(
    y_valid,
    deep_valid_prediction
)

deep_train_ba = balanced_accuracy_score(
    y_train,
    deep_train_prediction
)

deep_valid_ba = balanced_accuracy_score(
    y_valid,
    deep_valid_prediction
)

# Save results
deep_results = pd.DataFrame([{
    "model": "High capacity",
    "max_depth": 12,
    "rounds_completed": deep_rounds_completed,
    "best_iteration": deep_best_iteration,
    "minimum_validation_logloss":
        deep_validation_loss[deep_best_iteration],
    "training_mcc": deep_train_mcc,
    "validation_mcc": deep_valid_mcc,
    "training_validation_mcc_gap":
        deep_train_mcc - deep_valid_mcc,
    "training_balanced_accuracy": deep_train_ba,
    "validation_balanced_accuracy": deep_valid_ba
}])

display(deep_results)

deep_results.to_csv(
    "ga_deep_best_iteration_results.csv",
    index=False
)

deep_model.save_model(
    "ga_deep_best_iteration_model.json"
)

# Plot the learning curves
plt.figure(figsize=(9, 5))

plt.plot(
    range(deep_rounds_completed),
    deep_training_loss,
    label="Training log-loss"
)

plt.plot(
    range(deep_rounds_completed),
    deep_validation_loss,
    label="Validation log-loss"
)

plt.axvline(
    deep_best_iteration,
    linestyle="--",
    label=f"Best iteration: {deep_best_iteration}"
)

plt.xlabel("Boosting iteration")
plt.ylabel("Log-loss")
plt.title(
    "High-Capacity GA Model: "
    "Training and Validation Log-loss"
)
plt.legend()
plt.tight_layout()

plt.savefig(
    "ga_deep_best_iteration_learning_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print("ga_deep_best_iteration_results.csv")
print("ga_deep_best_iteration_model.json")
print("ga_deep_best_iteration_learning_curve.png")

In [ ]:
# Compare shallow, current and high-capacity models

capacity_comparison = pd.DataFrame([
    {
        "model": "Very shallow",
        "max_depth": 2,
        "best_iteration": shallow_final_best_iteration,
        "minimum_validation_logloss":
            shallow_final_valid_loss[
                shallow_final_best_iteration
            ],
        "training_mcc": shallow_final_train_mcc,
        "validation_mcc": shallow_final_valid_mcc,
        "training_validation_mcc_gap":
            shallow_final_train_mcc
            - shallow_final_valid_mcc,
        "validation_balanced_accuracy":
            shallow_final_valid_ba
    },
    {
        "model": "Current",
        "max_depth": 6,
        "best_iteration": best_iteration,
        "minimum_validation_logloss":
            validation_loss[best_iteration],
        "training_mcc": train_mcc_best,
        "validation_mcc": valid_mcc_best,
        "training_validation_mcc_gap":
            train_mcc_best - valid_mcc_best,
        "validation_balanced_accuracy":
            valid_ba_best
    },
    {
        "model": "High capacity",
        "max_depth": 12,
        "best_iteration": deep_best_iteration,
        "minimum_validation_logloss":
            deep_validation_loss[
                deep_best_iteration
            ],
        "training_mcc": deep_train_mcc,
        "validation_mcc": deep_valid_mcc,
        "training_validation_mcc_gap":
            deep_train_mcc - deep_valid_mcc,
        "validation_balanced_accuracy":
            deep_valid_ba
    }
])

capacity_comparison.to_csv(
    "ga_model_capacity_comparison.csv",
    index=False
)

display(capacity_comparison)

print("\nSaved:")
print("ga_model_capacity_comparison.csv")

In [ ]:
# Plot training and validation MCC across model capacities

plt.figure(figsize=(8, 5))

plt.plot(
    capacity_comparison["max_depth"],
    capacity_comparison["training_mcc"],
    marker="o",
    linewidth=2,
    label="Training MCC"
)

plt.plot(
    capacity_comparison["max_depth"],
    capacity_comparison["validation_mcc"],
    marker="o",
    linewidth=2,
    label="Validation MCC"
)

for _, row in capacity_comparison.iterrows():
    plt.text(
        row["max_depth"],
        row["training_mcc"] + 0.01,
        f'{row["training_mcc"]:.3f}',
        ha="center",
        fontsize=9
    )

    plt.text(
        row["max_depth"],
        row["validation_mcc"] - 0.025,
        f'{row["validation_mcc"]:.3f}',
        ha="center",
        fontsize=9
    )

plt.xticks(
    capacity_comparison["max_depth"],
    ["Very shallow\nDepth 2", "Current\nDepth 6", "High capacity\nDepth 12"]
)

plt.xlabel("Model capacity")
plt.ylabel("MCC")
plt.title("Effect of XGBoost Model Capacity on MCC")
plt.ylim(0.60, 1.02)
plt.legend()
plt.tight_layout()

plt.savefig(
    "ga_model_capacity_mcc_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Saved: ga_model_capacity_mcc_comparison.png")

In [ ]:
# Load the mRMR-selected descriptors

mrmr_features_data = pd.read_csv(
    "final_mordred_mrmr_300_descriptors.csv"
)

# Remove an unwanted saved index column if present
mrmr_features_data = mrmr_features_data.loc[
    :,
    ~mrmr_features_data.columns.str.startswith("Unnamed")
]

# Extract descriptor names
if "descriptor" in mrmr_features_data.columns:
    mrmr_features = (
        mrmr_features_data["descriptor"]
        .dropna()
        .tolist()
    )
else:
    mrmr_features = (
        mrmr_features_data.iloc[:, -1]
        .dropna()
        .tolist()
    )

# Full Mordred descriptor list
full_features = X_train.columns.tolist()

# Check that the mRMR descriptors are available
missing_mrmr_features = [
    feature
    for feature in mrmr_features
    if feature not in X_train.columns
    or feature not in X_valid.columns
]

print("Full Mordred descriptors:", len(full_features))
print("mRMR-selected descriptors:", len(mrmr_features))
print("GA-selected descriptors:", len(ga_features))
print("Missing mRMR descriptors:", len(missing_mrmr_features))

if len(missing_mrmr_features) == 0:
    print("All feature sets are ready for comparison.")
else:
    print(
        "Missing descriptors:",
        missing_mrmr_features[:10]
    )

In [ ]:
# Compare the three feature sets using their best iteration

feature_sets = {
    "Full Mordred": full_features,
    "mRMR": mrmr_features
}

feature_selection_results = []
feature_selection_models = {}

for method, features in feature_sets.items():

    print("Training:", method)

    model = XGBClassifier(
        objective="binary:logistic",
        n_estimators=3000,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=3,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_alpha=0.10,
        reg_lambda=1.00,
        tree_method="hist",
        eval_metric="logloss",
        early_stopping_rounds=200,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train[features],
        y_train,
        sample_weight=train_weights,
        eval_set=[
            (X_train[features], y_train),
            (X_valid[features], y_valid)
        ],
        verbose=False
    )

    history = model.evals_result()

    validation_logloss = history[
        "validation_1"
    ]["logloss"]

    method_best_iteration = model.best_iteration
    method_rounds_completed = len(validation_logloss)

    # Predictions using the best iteration
    train_probability = model.predict_proba(
        X_train[features],
        iteration_range=(
            0,
            method_best_iteration + 1
        )
    )[:, 1]

    valid_probability = model.predict_proba(
        X_valid[features],
        iteration_range=(
            0,
            method_best_iteration + 1
        )
    )[:, 1]

    train_prediction = (
        train_probability >= 0.50
    ).astype(int)

    valid_prediction = (
        valid_probability >= 0.50
    ).astype(int)

    train_mcc = matthews_corrcoef(
        y_train,
        train_prediction
    )

    valid_mcc = matthews_corrcoef(
        y_valid,
        valid_prediction
    )

    train_ba = balanced_accuracy_score(
        y_train,
        train_prediction
    )

    valid_ba = balanced_accuracy_score(
        y_valid,
        valid_prediction
    )

    feature_selection_results.append({
        "feature_selection": method,
        "number_of_features": len(features),
        "rounds_completed": method_rounds_completed,
        "best_iteration": method_best_iteration,
        "minimum_validation_logloss":
            validation_logloss[
                method_best_iteration
            ],
        "training_mcc": train_mcc,
        "validation_mcc": valid_mcc,
        "training_validation_mcc_gap":
            train_mcc - valid_mcc,
        "training_balanced_accuracy":
            train_ba,
        "validation_balanced_accuracy":
            valid_ba
    })

    feature_selection_models[method] = model

    model_name = (
        method.lower()
        .replace(" ", "_")
    )

    model.save_model(
        f"{model_name}_best_iteration_model.json"
    )

# Add the GA result already obtained
feature_selection_results.append({
    "feature_selection": "Genetic Algorithm",
    "number_of_features": len(ga_features),
    "rounds_completed": rounds_completed,
    "best_iteration": best_iteration,
    "minimum_validation_logloss":
        validation_loss[best_iteration],
    "training_mcc": train_mcc_best,
    "validation_mcc": valid_mcc_best,
    "training_validation_mcc_gap":
        train_mcc_best - valid_mcc_best,
    "training_balanced_accuracy":
        train_ba_best,
    "validation_balanced_accuracy":
        valid_ba_best
})

best_iteration_feature_comparison = pd.DataFrame(
    feature_selection_results
)

best_iteration_feature_comparison.to_csv(
    "best_iteration_feature_selection_validation_comparison.csv",
    index=False
)

display(best_iteration_feature_comparison)

print("\nSaved:")
print(
    "best_iteration_feature_selection_validation_comparison.csv"
)
print("full_mordred_best_iteration_model.json")
print("mrmr_best_iteration_model.json")

In [ ]:
# Load the test data

X_test = pd.read_pickle(
    "mordred_test_features_filtered.pkl"
)

y_test = pd.read_pickle(
    "mordred_test_labels.pkl"
)

# Check that all feature sets are present in the test data
missing_full_test = [
    feature for feature in full_features
    if feature not in X_test.columns
]

missing_mrmr_test = [
    feature for feature in mrmr_features
    if feature not in X_test.columns
]

missing_ga_test = [
    feature for feature in ga_features
    if feature not in X_test.columns
]

print("Test features shape:", X_test.shape)
print("Test labels:", len(y_test))
print("Missing Full Mordred descriptors:", len(missing_full_test))
print("Missing mRMR descriptors:", len(missing_mrmr_test))
print("Missing GA descriptors:", len(missing_ga_test))

if (
    len(missing_full_test) == 0
    and len(missing_mrmr_test) == 0
    and len(missing_ga_test) == 0
):
    print("The test data is ready for evaluation.")

In [ ]:
# Refit each model using training and validation data

# Combine training and validation data
X_train_valid = pd.concat(
    [X_train, X_valid],
    axis=0
)

y_train_valid = pd.concat(
    [
        pd.Series(y_train),
        pd.Series(y_valid)
    ],
    axis=0
)

# Calculate balanced weights
train_valid_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_valid
)

# Best number of trees selected using validation loss
selected_iterations = {
    "Full Mordred": 1429,
    "mRMR": 1423,
    "Genetic Algorithm": 1548
}

feature_sets_final = {
    "Full Mordred": full_features,
    "mRMR": mrmr_features,
    "Genetic Algorithm": ga_features
}

fair_best_iteration_results = []
fair_best_iteration_predictions = []

for method, features in feature_sets_final.items():

    print("Training final model:", method)

    number_of_trees = selected_iterations[method] + 1

    final_model = XGBClassifier(
        objective="binary:logistic",
        n_estimators=number_of_trees,
        learning_rate=0.05,
        max_depth=6,
        min_child_weight=3,
        subsample=0.80,
        colsample_bytree=0.80,
        reg_alpha=0.10,
        reg_lambda=1.00,
        tree_method="hist",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    final_model.fit(
        X_train_valid[features],
        y_train_valid,
        sample_weight=train_valid_weights,
        verbose=False
    )

    test_probability = final_model.predict_proba(
        X_test[features]
    )[:, 1]

    test_prediction = (
        test_probability >= 0.50
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        test_prediction
    ).ravel()

    specificity = tn / (tn + fp)

    fair_best_iteration_results.append({
        "feature_selection": method,
        "number_of_features": len(features),
        "selected_iteration":
            selected_iterations[method],
        "number_of_trees": number_of_trees,
        "test_accuracy": accuracy_score(
            y_test,
            test_prediction
        ),
        "test_balanced_accuracy":
            balanced_accuracy_score(
                y_test,
                test_prediction
            ),
        "test_mcc": matthews_corrcoef(
            y_test,
            test_prediction
        ),
        "test_roc_auc": roc_auc_score(
            y_test,
            test_probability
        ),
        "test_average_precision":
            average_precision_score(
                y_test,
                test_probability
            ),
        "test_precision": precision_score(
            y_test,
            test_prediction,
            zero_division=0
        ),
        "test_recall": recall_score(
            y_test,
            test_prediction,
            zero_division=0
        ),
        "test_specificity": specificity,
        "test_f1": f1_score(
            y_test,
            test_prediction,
            zero_division=0
        ),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    })

    fair_best_iteration_predictions.append(
        pd.DataFrame({
            "feature_selection": method,
            "true_label": np.asarray(y_test),
            "predicted_probability":
                test_probability,
            "predicted_label":
                test_prediction
        })
    )

    model_name = (
        method.lower()
        .replace(" ", "_")
    )

    final_model.save_model(
        f"{model_name}_fair_best_iteration_model.json"
    )

fair_best_iteration_comparison = pd.DataFrame(
    fair_best_iteration_results
)

fair_best_iteration_prediction_table = pd.concat(
    fair_best_iteration_predictions,
    ignore_index=True
)

fair_best_iteration_comparison.to_csv(
    "fair_best_iteration_test_comparison.csv",
    index=False
)

fair_best_iteration_prediction_table.to_csv(
    "fair_best_iteration_test_predictions.csv",
    index=False
)

display(fair_best_iteration_comparison)

print("\nSaved:")
print("fair_best_iteration_test_comparison.csv")
print("fair_best_iteration_test_predictions.csv")

In [ ]:
# Compare fixed 1000-round models with best-iteration refitted models

fixed_1000_results = pd.read_csv(
    "final_fair_feature_selection_comparison.csv"
)

best_iteration_refit_results = pd.read_csv(
    "fair_best_iteration_test_comparison.csv"
)

# Select required columns from the original comparison
fixed_1000_results = fixed_1000_results[[
    "feature_selection",
    "number_of_features",
    "mcc",
    "balanced_accuracy",
    "roc_auc"
]].rename(columns={
    "mcc": "fixed_1000_test_mcc",
    "balanced_accuracy":
        "fixed_1000_test_balanced_accuracy",
    "roc_auc": "fixed_1000_test_roc_auc"
})

# Select required columns from the new comparison
best_iteration_refit_results = (
    best_iteration_refit_results[[
        "feature_selection",
        "selected_iteration",
        "number_of_trees",
        "test_mcc",
        "test_balanced_accuracy",
        "test_roc_auc"
    ]]
)

# Combine both analyses
final_iteration_comparison = fixed_1000_results.merge(
    best_iteration_refit_results,
    on="feature_selection",
    how="inner"
)

# Calculate the changes
final_iteration_comparison["mcc_change"] = (
    final_iteration_comparison["test_mcc"]
    - final_iteration_comparison[
        "fixed_1000_test_mcc"
    ]
)

final_iteration_comparison[
    "balanced_accuracy_change"
] = (
    final_iteration_comparison[
        "test_balanced_accuracy"
    ]
    - final_iteration_comparison[
        "fixed_1000_test_balanced_accuracy"
    ]
)

final_iteration_comparison["roc_auc_change"] = (
    final_iteration_comparison["test_roc_auc"]
    - final_iteration_comparison[
        "fixed_1000_test_roc_auc"
    ]
)

final_iteration_comparison.to_csv(
    "final_fixed_1000_vs_best_iteration_refit_comparison.csv",
    index=False
)

display(final_iteration_comparison)

print("\nSaved:")
print(
    "final_fixed_1000_vs_best_iteration_refit_comparison.csv"
)

In [ ]:
# Create separate GA scripts for five seeds
additional_seeds = [7, 21, 84, 123, 99]

with open("full_ga_run.py", "r") as file:
    original_script_lines = file.readlines()

seed_run_plan = []

for seed in additional_seeds:
    new_script_lines = []
    for line in original_script_lines:
        stripped_line = line.strip()
        indentation = line[:len(line) - len(line.lstrip())]

        if stripped_line == (
            'results_folder = main_folder / "full_ga_results"'
        ):
            new_line = (
                indentation
                + f'results_folder = main_folder / '
                + f'"full_ga_results_seed_{seed}"\n'
            )
        elif stripped_line == "random.seed(42)":
            new_line = indentation + f"random.seed({seed})\n"
        elif stripped_line == "np.random.seed(42)":
            new_line = indentation + f"np.random.seed({seed})\n"
        else:
            new_line = line
        new_script_lines.append(new_line)

    script_name = f"full_ga_run_seed_{seed}.py"
    result_folder = f"full_ga_results_seed_{seed}"

    with open(script_name, "w") as file:
        file.writelines(new_script_lines)

    seed_run_plan.append({
        "ga_seed": seed,
        "xgboost_random_state": 42,
        "script_name": script_name,
        "result_folder": result_folder,
        "population_size": 40,
        "generations": 30
    })

ga_seed_run_plan = pd.DataFrame(seed_run_plan)
ga_seed_run_plan.to_csv("ga_seed_run_plan.csv", index=False)
display(ga_seed_run_plan)

# Check the changed lines in each script
for seed in additional_seeds:
    script_name = f"full_ga_run_seed_{seed}.py"
    print(f"\nChecking {script_name}:")
    with open(script_name, "r") as file:
        for line_number, line in enumerate(file, start=1):
            if (
                "results_folder =" in line
                or line.strip() == f"random.seed({seed})"
                or line.strip() == f"np.random.seed({seed})"
                or '"random_state": 42' in line
            ):
                print(f"{line_number}: {line.rstrip()}")

print("\nSaved:")
print("ga_seed_run_plan.csv")
print("Five GA scripts")

In [ ]:
# Start the GA run for seed 21

import sys
import subprocess
import pandas as pd

from pathlib import Path

seed = 21

script_file = Path(f"full_ga_run_seed_{seed}.py")
result_folder = Path(f"full_ga_results_seed_{seed}")
history_file = result_folder / "full_ga_history.csv"
log_file = Path(f"full_ga_seed_{seed}.log")
pid_file = Path(f"full_ga_seed_{seed}_pid.txt")

# Check the latest completed generation
latest_generation = -1

if history_file.exists():
    history_data = pd.read_csv(history_file)

    if "generation" in history_data.columns:
        latest_generation = int(
            history_data["generation"].max()
        )
    elif "gen" in history_data.columns:
        latest_generation = int(
            history_data["gen"].max()
        )

# Check for an active seed 21 process
process_list = subprocess.run(
    ["ps", "-eo", "pid=,cmd="],
    capture_output=True,
    text=True
).stdout

active_processes = [
    line
    for line in process_list.splitlines()
    if script_file.name in line
    and "python" in line
]

if latest_generation >= 30:
    print("Seed 21 GA run is already complete.")

elif active_processes:
    print("Seed 21 GA run is already running.")

    for process in active_processes:
        print(process)

elif not script_file.exists():
    print("Script not found:", script_file)

else:
    log_output = open(log_file, "a")

    process = subprocess.Popen(
        [sys.executable, str(script_file)],
        stdout=log_output,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

    pid_file.write_text(str(process.pid))

    print("Seed 21 GA run started.")
    print("Process ID:", process.pid)
    print("Log file:", log_file)
    print("Result folder:", result_folder)

In [ ]:
import sys

In [ ]:
# Start the GA run for seed 84

seed = 84

script_file = Path(f"full_ga_run_seed_{seed}.py")
result_folder = Path(f"full_ga_results_seed_{seed}")
history_file = result_folder / "full_ga_history.csv"
log_file = Path(f"full_ga_seed_{seed}.log")
pid_file = Path(f"full_ga_seed_{seed}_pid.txt")

latest_generation = -1

if history_file.exists():
    history_data = pd.read_csv(history_file)

    if "generation" in history_data.columns:
        latest_generation = int(
            history_data["generation"].max()
        )
    elif "gen" in history_data.columns:
        latest_generation = int(
            history_data["gen"].max()
        )

process_list = subprocess.run(
    ["ps", "-eo", "pid=,cmd="],
    capture_output=True,
    text=True
).stdout

active_processes = [
    line
    for line in process_list.splitlines()
    if script_file.name in line
    and "python" in line
]

if latest_generation >= 30:
    print("Seed 84 GA run is already complete.")

elif active_processes:
    print("Seed 84 GA run is already running.")

    for process in active_processes:
        print(process)

elif not script_file.exists():
    print("Script not found:", script_file)

else:
    log_output = open(log_file, "a")

    process = subprocess.Popen(
        [sys.executable, str(script_file)],
        stdout=log_output,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

    pid_file.write_text(str(process.pid))

    print("Seed 84 GA run started.")
    print("Process ID:", process.pid)
    print("Log file:", log_file)
    print("Result folder:", result_folder)

In [ ]:
# Start the GA run for seed 123
import sys
import pandas as pd
import subprocess
from pathlib import Path

seed = 123
script_file = Path(f"full_ga_run_seed_{seed}.py")
result_folder = Path(f"full_ga_results_seed_{seed}")
history_file = result_folder / "full_ga_history.csv"
log_file = Path(f"full_ga_seed_{seed}.log")
pid_file = Path(f"full_ga_seed_{seed}_pid.txt")

latest_generation = -1
if history_file.exists():
    history_data = pd.read_csv(history_file)
    if "generation" in history_data.columns:
        latest_generation = int(history_data["generation"].max())
    elif "gen" in history_data.columns:
        latest_generation = int(history_data["gen"].max())

process_list = subprocess.run(
    ["ps", "-eo", "pid=,cmd="],
    capture_output=True,
    text=True
).stdout

active_processes = [
    line
    for line in process_list.splitlines()
    if script_file.name in line
    and "python" in line
]

if latest_generation >= 30:
    print("Seed 123 GA run is already complete.")
elif active_processes:
    print("Seed 123 GA run is already running.")
    for process in active_processes:
        print(process)
elif not script_file.exists():
    print("Script not found:", script_file)
else:
    log_output = open(log_file, "a")
    process = subprocess.Popen(
        [sys.executable, str(script_file)],
        stdout=log_output,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )
    pid_file.write_text(str(process.pid))
    print("Seed 123 GA run started.")
    print("Process ID:", process.pid)
    print("Log file:", log_file)
    print("Result folder:", result_folder)

In [ ]:
# Start the GA run for seed 99
import sys
import pandas as pd
import subprocess
from pathlib import Path

seed = 99
script_file = Path(f"full_ga_run_seed_{seed}.py")
result_folder = Path(f"full_ga_results_seed_{seed}")
history_file = result_folder / "full_ga_history.csv"
log_file = Path(f"full_ga_seed_{seed}.log")
pid_file = Path(f"full_ga_seed_{seed}_pid.txt")

latest_generation = -1
if history_file.exists():
    history_data = pd.read_csv(history_file)
    if "generation" in history_data.columns:
        latest_generation = int(history_data["generation"].max())
    elif "gen" in history_data.columns:
        latest_generation = int(history_data["gen"].max())

process_list = subprocess.run(
    ["ps", "-eo", "pid=,cmd="],
    capture_output=True,
    text=True
).stdout

active_processes = [
    line
    for line in process_list.splitlines()
    if script_file.name in line
    and "python" in line
]

if latest_generation >= 30:
    print("Seed 99 GA run is already complete.")
elif active_processes:
    print("Seed 99 GA run is already running.")
    for process in active_processes:
        print(process)
elif not script_file.exists():
    print("Script not found:", script_file)
else:
    log_output = open(log_file, "a")
    process = subprocess.Popen(
        [sys.executable, str(script_file)],
        stdout=log_output,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )
    pid_file.write_text(str(process.pid))
    print("Seed 99 GA run started.")
    print("Process ID:", process.pid)
    print("Log file:", log_file)
    print("Result folder:", result_folder)

In [ ]:
# Find the file containing the continuous docking-score targets

print(
    "split_assignments.csv columns:\n",
    pd.read_csv(
        "split_assignments.csv",
        nrows=5
    ).columns.tolist()
)

target_file_matches = []

for file_path in Path(".").rglob("*.csv"):

    try:
        columns = pd.read_csv(
            file_path,
            nrows=5
        ).columns.tolist()

        matching_columns = [
            column
            for column in columns
            if (
                "jnk3" in column.lower()
                or "gsk3" in column.lower()
            )
        ]

        if matching_columns:
            target_file_matches.append({
                "file": str(file_path),
                "matching_columns": matching_columns,
                "all_columns": columns
            })

    except Exception:
        pass

target_file_matches = pd.DataFrame(
    target_file_matches
)

display(target_file_matches)

In [ ]:
# Load and align the continuous docking-score targets

# Load the scaffold split information
split_data = pd.read_csv(
    "split_assignments.csv"
).reset_index().rename(
    columns={"index": "row_index"}
)

# Load the cleaned docking scores
score_data = pd.read_csv(
    "model_dataset.csv"
)[[
    "canonical_smiles",
    "jnk3_score",
    "gsk3b_score"
]]

target_columns = [
    "jnk3_score",
    "gsk3b_score"
]

# Check for duplicate molecules
print(
    "Duplicate molecules in split data:",
    split_data["canonical_smiles"].duplicated().sum()
)

print(
    "Duplicate molecules in score data:",
    score_data["canonical_smiles"].duplicated().sum()
)

# Match docking scores with the scaffold split
target_data = split_data.merge(
    score_data,
    on="canonical_smiles",
    how="left",
    validate="one_to_one"
)

target_data = target_data.set_index(
    "row_index"
)

# Select targets using the same rows as the feature datasets
y_train_regression = target_data.loc[
    X_train.index,
    target_columns
].copy()

y_valid_regression = target_data.loc[
    X_valid.index,
    target_columns
].copy()

# Check the aligned targets
print(
    "Training target shape:",
    y_train_regression.shape
)

print(
    "Validation target shape:",
    y_valid_regression.shape
)

print(
    "Missing training target values:",
    y_train_regression.isna().sum().sum()
)

print(
    "Missing validation target values:",
    y_valid_regression.isna().sum().sum()
)

print("\nTraining target summary:")
display(y_train_regression.describe())

print("\nValidation target summary:")
display(y_valid_regression.describe())

In [ ]:
# Train the neural network regression model

import torch
import torch.nn as nn
import joblib

from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Set random seeds
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

# Scale the GA-selected descriptors
feature_scaler = StandardScaler()

X_train_reg_scaled = feature_scaler.fit_transform(
    X_train[ga_features]
)

X_valid_reg_scaled = feature_scaler.transform(
    X_valid[ga_features]
)

# Scale both docking-score targets
target_scaler = StandardScaler()

y_train_reg_scaled = target_scaler.fit_transform(
    y_train_regression
)

y_valid_reg_scaled = target_scaler.transform(
    y_valid_regression
)

# Convert the training data to tensors
X_train_reg_tensor = torch.tensor(
    X_train_reg_scaled,
    dtype=torch.float32
)

y_train_reg_tensor = torch.tensor(
    y_train_reg_scaled,
    dtype=torch.float32
)

X_valid_reg_tensor = torch.tensor(
    X_valid_reg_scaled,
    dtype=torch.float32
).to(device)

y_valid_reg_tensor = torch.tensor(
    y_valid_reg_scaled,
    dtype=torch.float32
).to(device)

# Create training batches
train_reg_dataset = TensorDataset(
    X_train_reg_tensor,
    y_train_reg_tensor
)

data_generator = torch.Generator()
data_generator.manual_seed(SEED)

train_reg_loader = DataLoader(
    train_reg_dataset,
    batch_size=512,
    shuffle=True,
    generator=data_generator
)

# Define the neural network
class MLPRegressor(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.network(x)


regression_model = MLPRegressor(
    input_dim=len(ga_features)
).to(device)

loss_function = nn.MSELoss()

optimizer = torch.optim.Adam(
    regression_model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

# Early-stopping settings
maximum_epochs = 300
patience = 20

best_validation_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
best_model_state = None

training_losses = []
validation_losses = []

# Train the model
for epoch in range(maximum_epochs):

    regression_model.train()
    total_training_loss = 0.0

    for batch_features, batch_targets in train_reg_loader:

        batch_features = batch_features.to(device)
        batch_targets = batch_targets.to(device)

        optimizer.zero_grad()

        predictions = regression_model(
            batch_features
        )

        loss = loss_function(
            predictions,
            batch_targets
        )

        loss.backward()
        optimizer.step()

        total_training_loss += (
            loss.item() * batch_features.size(0)
        )

    average_training_loss = (
        total_training_loss
        / len(train_reg_dataset)
    )

    training_losses.append(
        average_training_loss
    )

    # Calculate validation loss
    regression_model.eval()

    with torch.no_grad():

        validation_output = regression_model(
            X_valid_reg_tensor
        )

        validation_loss = loss_function(
            validation_output,
            y_valid_reg_tensor
        ).item()

    validation_losses.append(
        validation_loss
    )

    # Keep the epoch with the lowest validation loss
    if validation_loss < best_validation_loss:

        best_validation_loss = validation_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        best_model_state = {
            name: value.detach().cpu().clone()
            for name, value
            in regression_model.state_dict().items()
        }

    else:
        epochs_without_improvement += 1

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch} | "
            f"Train loss: {average_training_loss:.5f} | "
            f"Valid loss: {validation_loss:.5f}"
        )

    if epochs_without_improvement >= patience:

        print(
            f"\nEarly stopping at epoch {epoch}."
        )

        print("Best epoch:", best_epoch)
        break

# Load the best epoch
regression_model.load_state_dict(
    best_model_state
)

regression_model = regression_model.to(device)
regression_model.eval()

# Make predictions using the best model
with torch.no_grad():

    train_reg_predictions_scaled = regression_model(
        torch.tensor(
            X_train_reg_scaled,
            dtype=torch.float32
        ).to(device)
    ).cpu().numpy()

    valid_reg_predictions_scaled = regression_model(
        X_valid_reg_tensor
    ).cpu().numpy()

# Convert predictions back to docking-score units
train_reg_predictions = target_scaler.inverse_transform(
    train_reg_predictions_scaled
)

valid_reg_predictions = target_scaler.inverse_transform(
    valid_reg_predictions_scaled
)

train_reg_true = y_train_regression.to_numpy()
valid_reg_true = y_valid_regression.to_numpy()

# Calculate results for each target
regression_results = []

for target_index, target_name in enumerate(
    target_columns
):

    train_mae = mean_absolute_error(
        train_reg_true[:, target_index],
        train_reg_predictions[:, target_index]
    )

    valid_mae = mean_absolute_error(
        valid_reg_true[:, target_index],
        valid_reg_predictions[:, target_index]
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            train_reg_true[:, target_index],
            train_reg_predictions[:, target_index]
        )
    )

    valid_rmse = np.sqrt(
        mean_squared_error(
            valid_reg_true[:, target_index],
            valid_reg_predictions[:, target_index]
        )
    )

    train_r2 = r2_score(
        train_reg_true[:, target_index],
        train_reg_predictions[:, target_index]
    )

    valid_r2 = r2_score(
        valid_reg_true[:, target_index],
        valid_reg_predictions[:, target_index]
    )

    regression_results.append({
        "target": target_name,
        "number_of_features": len(ga_features),
        "epochs_completed": epoch + 1,
        "best_epoch": best_epoch,
        "minimum_validation_loss":
            best_validation_loss,
        "training_mae": train_mae,
        "validation_mae": valid_mae,
        "training_rmse": train_rmse,
        "validation_rmse": valid_rmse,
        "training_r2": train_r2,
        "validation_r2": valid_r2
    })

regression_results = pd.DataFrame(
    regression_results
)

display(regression_results)

# Save validation predictions
regression_validation_predictions = pd.DataFrame({
    "jnk3_true": valid_reg_true[:, 0],
    "jnk3_predicted":
        valid_reg_predictions[:, 0],
    "gsk3b_true": valid_reg_true[:, 1],
    "gsk3b_predicted":
        valid_reg_predictions[:, 1]
})

regression_results.to_csv(
    "ga_nn_regression_results.csv",
    index=False
)

regression_validation_predictions.to_csv(
    "ga_nn_regression_validation_predictions.csv",
    index=False
)

# Save the model and scalers
torch.save(
    {
        "model_state_dict": best_model_state,
        "input_dimension": len(ga_features),
        "best_epoch": best_epoch,
        "target_columns": target_columns,
        "ga_features": ga_features
    },
    "ga_nn_regression_model.pt"
)

joblib.dump(
    feature_scaler,
    "ga_nn_regression_feature_scaler.pkl"
)

joblib.dump(
    target_scaler,
    "ga_nn_regression_target_scaler.pkl"
)

# Plot the learning curves
plt.figure(figsize=(9, 5))

plt.plot(
    training_losses,
    label="Training loss"
)

plt.plot(
    validation_losses,
    label="Validation loss"
)

plt.axvline(
    best_epoch,
    linestyle="--",
    label=f"Best epoch: {best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("Standardised MSE loss")
plt.title(
    "Neural Network Regression Learning Curve"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    "ga_nn_regression_learning_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print("ga_nn_regression_results.csv")
print("ga_nn_regression_validation_predictions.csv")
print("ga_nn_regression_model.pt")
print("ga_nn_regression_feature_scaler.pkl")
print("ga_nn_regression_target_scaler.pkl")
print("ga_nn_regression_learning_curve.png")

In [ ]:
# Collect the completed GA seed results

seed_folders = {
    42: Path("full_ga_results")
}

# Find the additional seed result folders
for folder in Path(".").glob(
    "full_ga_results_seed_*"
):
    try:
        seed_number = int(
            folder.name.split("_")[-1]
        )

        seed_folders[seed_number] = folder

    except ValueError:
        pass

seed_results = []

for seed, folder in sorted(
    seed_folders.items()
):

    history_file = (
        folder / "full_ga_history.csv"
    )

    features_file = (
        folder / "full_ga_best_features.csv"
    )

    if (
        not history_file.exists()
        or not features_file.exists()
    ):
        continue

    history = pd.read_csv(
        history_file
    )

    # Find the generation column
    generation_columns = [
        column
        for column in history.columns
        if (
            column.lower() == "generation"
            or column.lower() == "gen"
        )
    ]

    if not generation_columns:
        continue

    generation_column = (
        generation_columns[0]
    )

    latest_generation = int(
        history[generation_column].max()
    )

    # Only include completed runs
    if latest_generation < 30:
        continue

    final_row = history.loc[
        history[generation_column].idxmax()
    ]

    # Find the best MCC column
    mcc_columns = [
        column
        for column in history.columns
        if (
            "best" in column.lower()
            and "mcc" in column.lower()
        )
    ]

    if not mcc_columns:
        continue

    best_mcc_column = mcc_columns[0]

    best_validation_mcc = float(
        history[best_mcc_column].max()
    )

    # Count the selected descriptors
    selected_features = pd.read_csv(
        features_file
    )

    selected_features = (
        selected_features.loc[
            :,
            ~selected_features.columns.str.startswith(
                "Unnamed"
            )
        ]
    )

    number_of_descriptors = len(
        selected_features
    )

    # Read optional information from the history
    unique_columns = [
        column
        for column in history.columns
        if "unique" in column.lower()
    ]

    elapsed_columns = [
        column
        for column in history.columns
        if (
            "elapsed" in column.lower()
            and "second" in column.lower()
        )
    ]

    unique_subsets = (
        int(final_row[unique_columns[0]])
        if unique_columns
        else np.nan
    )

    elapsed_minutes = (
        float(final_row[elapsed_columns[0]])
        / 60
        if elapsed_columns
        else np.nan
    )

    seed_results.append({
        "seed": seed,
        "final_generation":
            latest_generation,
        "best_validation_mcc":
            best_validation_mcc,
        "selected_descriptors":
            number_of_descriptors,
        "unique_subsets_evaluated":
            unique_subsets,
        "elapsed_minutes":
            elapsed_minutes,
        "result_folder":
            str(folder)
    })

ga_seed_summary = pd.DataFrame(
    seed_results
).sort_values(
    "seed"
).reset_index(
    drop=True
)

ga_seed_summary.to_csv(
    "ga_completed_seed_summary.csv",
    index=False
)

display(ga_seed_summary)

print(
    "\nNumber of completed GA seeds:",
    len(ga_seed_summary)
)

print(
    "Completed seeds:",
    ga_seed_summary["seed"].tolist()
)

print(
    "\nSaved: ga_completed_seed_summary.csv"
)

In [ ]:
# Summarise GA performance across seeds

ga_seed_statistics = pd.DataFrame([{
    "number_of_seeds": len(ga_seed_summary),

    "mean_validation_mcc":
        ga_seed_summary[
            "best_validation_mcc"
        ].mean(),

    "standard_deviation_validation_mcc":
        ga_seed_summary[
            "best_validation_mcc"
        ].std(ddof=1),

    "minimum_validation_mcc":
        ga_seed_summary[
            "best_validation_mcc"
        ].min(),

    "maximum_validation_mcc":
        ga_seed_summary[
            "best_validation_mcc"
        ].max(),

    "mean_selected_descriptors":
        ga_seed_summary[
            "selected_descriptors"
        ].mean(),

    "standard_deviation_selected_descriptors":
        ga_seed_summary[
            "selected_descriptors"
        ].std(ddof=1),

    "minimum_selected_descriptors":
        ga_seed_summary[
            "selected_descriptors"
        ].min(),

    "maximum_selected_descriptors":
        ga_seed_summary[
            "selected_descriptors"
        ].max(),

    "best_seed":
        int(
            ga_seed_summary.loc[
                ga_seed_summary[
                    "best_validation_mcc"
                ].idxmax(),
                "seed"
            ]
        ),

    "worst_seed":
        int(
            ga_seed_summary.loc[
                ga_seed_summary[
                    "best_validation_mcc"
                ].idxmin(),
                "seed"
            ]
        )
}])

ga_seed_statistics.to_csv(
    "ga_seed_stability_statistics.csv",
    index=False
)

display(ga_seed_statistics)

print(
    "\nSaved: ga_seed_stability_statistics.csv"
)

In [ ]:
# Calculate descriptor overlap between GA seeds

seed_feature_sets = {}

for _, row in ga_seed_summary.iterrows():

    seed = int(row["seed"])
    result_folder = Path(row["result_folder"])

    features_file = (
        result_folder
        / "full_ga_best_features.csv"
    )

    feature_data = pd.read_csv(
        features_file
    )

    feature_data = feature_data.loc[
        :,
        ~feature_data.columns.str.startswith(
            "Unnamed"
        )
    ]

    if "descriptor" in feature_data.columns:
        selected_features = set(
            feature_data["descriptor"]
            .dropna()
            .astype(str)
        )
    else:
        selected_features = set(
            feature_data.iloc[:, 0]
            .dropna()
            .astype(str)
        )

    seed_feature_sets[seed] = selected_features

seeds = sorted(seed_feature_sets.keys())

jaccard_matrix = pd.DataFrame(
    index=seeds,
    columns=seeds,
    dtype=float
)

overlap_results = []

for seed_1 in seeds:
    for seed_2 in seeds:

        features_1 = seed_feature_sets[seed_1]
        features_2 = seed_feature_sets[seed_2]

        shared_features = (
            features_1.intersection(
                features_2
            )
        )

        combined_features = (
            features_1.union(
                features_2
            )
        )

        jaccard_score = (
            len(shared_features)
            / len(combined_features)
        )

        jaccard_matrix.loc[
            seed_1,
            seed_2
        ] = jaccard_score

        if seed_1 < seed_2:
            overlap_results.append({
                "seed_1": seed_1,
                "seed_2": seed_2,
                "features_seed_1":
                    len(features_1),
                "features_seed_2":
                    len(features_2),
                "shared_features":
                    len(shared_features),
                "jaccard_similarity":
                    jaccard_score
            })

pairwise_overlap = pd.DataFrame(
    overlap_results
)

pairwise_overlap.to_csv(
    "ga_seed_pairwise_descriptor_overlap.csv",
    index=False
)

jaccard_matrix.to_csv(
    "ga_seed_jaccard_matrix.csv"
)

display(pairwise_overlap)
display(jaccard_matrix)

print(
    "\nMean pairwise Jaccard similarity:",
    round(
        pairwise_overlap[
            "jaccard_similarity"
        ].mean(),
        4
    )
)

print(
    "Minimum pairwise Jaccard similarity:",
    round(
        pairwise_overlap[
            "jaccard_similarity"
        ].min(),
        4
    )
)

print(
    "Maximum pairwise Jaccard similarity:",
    round(
        pairwise_overlap[
            "jaccard_similarity"
        ].max(),
        4
    )
)

print("\nSaved:")
print("ga_seed_pairwise_descriptor_overlap.csv")
print("ga_seed_jaccard_matrix.csv")

# Final XGBoost Analysis Using GA Seed 84

GA seed 84 achieved the highest validation MCC across the six completed genetic-algorithm runs. This section therefore uses its selected descriptor subset to determine the optimal XGBoost boosting iteration using validation performance only. The test set remains unused until model and iteration selection are complete.

In [ ]:
# Prepare the seed 84 training and validation data

from pathlib import Path

seed_84_feature_file = Path(
    "full_ga_results_seed_84/"
    "full_ga_best_features.csv"
)

training_feature_file = Path(
    "mordred_training_features_filtered.pkl"
)

validation_feature_file = Path(
    "mordred_validation_features_filtered.pkl"
)

model_data_file = Path(
    "model_dataset.csv"
)

required_files = [
    seed_84_feature_file,
    training_feature_file,
    validation_feature_file,
    model_data_file
]

missing_files = [
    str(file)
    for file in required_files
    if not file.exists()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing files: {missing_files}"
    )

# Load the filtered Mordred feature matrices
X_train_full = pd.read_pickle(
    training_feature_file
).astype("float32")

X_valid_full = pd.read_pickle(
    validation_feature_file
).astype("float32")

# Load the classification labels
model_data_seed84 = pd.read_csv(
    model_data_file
)

unnamed_columns = [
    column
    for column in model_data_seed84.columns
    if column.startswith("Unnamed")
]

if len(unnamed_columns) == 1:

    saved_index_column = unnamed_columns[0]

    if model_data_seed84[
        saved_index_column
    ].is_unique:

        model_data_seed84 = (
            model_data_seed84.set_index(
                saved_index_column
            )
        )

    else:
        model_data_seed84 = (
            model_data_seed84.drop(
                columns=unnamed_columns
            )
        )

# Load the descriptors selected by seed 84
seed_84_feature_data = pd.read_csv(
    seed_84_feature_file
)

seed_84_feature_data = (
    seed_84_feature_data.loc[
        :,
        ~seed_84_feature_data.columns.str.startswith(
            "Unnamed"
        )
    ]
)

if (
    "descriptor"
    in seed_84_feature_data.columns
):
    descriptor_column = "descriptor"
else:
    descriptor_column = (
        seed_84_feature_data.columns[0]
    )

seed_84_descriptors = (
    seed_84_feature_data[
        descriptor_column
    ]
    .dropna()
    .astype(str)
    .tolist()
)

# Check the selected descriptors
missing_training_descriptors = [
    descriptor
    for descriptor in seed_84_descriptors
    if descriptor not in X_train_full.columns
]

missing_validation_descriptors = [
    descriptor
    for descriptor in seed_84_descriptors
    if descriptor not in X_valid_full.columns
]

if missing_training_descriptors:
    raise ValueError(
        "Descriptors missing from training data: "
        f"{missing_training_descriptors[:10]}"
    )

if missing_validation_descriptors:
    raise ValueError(
        "Descriptors missing from validation data: "
        f"{missing_validation_descriptors[:10]}"
    )

if (
    "dual_candidate"
    not in model_data_seed84.columns
):
    raise ValueError(
        "dual_candidate is missing from "
        "model_dataset.csv"
    )

# Keep only the seed 84 descriptors
X_train_seed84 = X_train_full[
    seed_84_descriptors
].copy()

X_valid_seed84 = X_valid_full[
    seed_84_descriptors
].copy()

# Align labels using the saved row indices
y_train_seed84 = (
    model_data_seed84.loc[
        X_train_seed84.index,
        "dual_candidate"
    ]
    .astype("int8")
)

y_valid_seed84 = (
    model_data_seed84.loc[
        X_valid_seed84.index,
        "dual_candidate"
    ]
    .astype("int8")
)

# Final checks
if X_train_seed84.isna().any().any():
    raise ValueError(
        "Missing values found in training features."
    )

if X_valid_seed84.isna().any().any():
    raise ValueError(
        "Missing values found in validation features."
    )

if y_train_seed84.isna().any():
    raise ValueError(
        "Missing values found in training labels."
    )

if y_valid_seed84.isna().any():
    raise ValueError(
        "Missing values found in validation labels."
    )

print(
    "Seed 84 selected descriptors:",
    len(seed_84_descriptors)
)

print(
    "Training feature shape:",
    X_train_seed84.shape
)

print(
    "Validation feature shape:",
    X_valid_seed84.shape
)

print(
    "\nTraining class counts:"
)

print(
    y_train_seed84.value_counts()
    .sort_index()
)

print(
    "\nValidation class counts:"
)

print(
    y_valid_seed84.value_counts()
    .sort_index()
)

print(
    "\nTraining and validation descriptor order matches:",
    X_train_seed84.columns.equals(
        X_valid_seed84.columns
    )
)

print(
    "Test set loaded:",
    False
)

In [ ]:
# Find the best boosting iteration for GA seed 84

# Calculate balanced training weights
seed84_train_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train_seed84
)

seed84_best_iteration_model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=3000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.10,
    reg_lambda=1.00,
    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=200,
    random_state=42,
    n_jobs=-1
)

# Train while monitoring training and validation loss
seed84_best_iteration_model.fit(
    X_train_seed84,
    y_train_seed84,
    sample_weight=seed84_train_weights,
    eval_set=[
        (
            X_train_seed84,
            y_train_seed84
        ),
        (
            X_valid_seed84,
            y_valid_seed84
        )
    ],
    verbose=False
)

# Get the learning history
seed84_history = (
    seed84_best_iteration_model.evals_result()
)

seed84_training_loss = (
    seed84_history[
        "validation_0"
    ]["logloss"]
)

seed84_validation_loss = (
    seed84_history[
        "validation_1"
    ]["logloss"]
)

seed84_best_iteration = (
    seed84_best_iteration_model.best_iteration
)

seed84_rounds_completed = len(
    seed84_validation_loss
)

seed84_last_iteration = (
    seed84_rounds_completed - 1
)

print(
    "Maximum boosting rounds:",
    3000
)

print(
    "Rounds completed:",
    seed84_rounds_completed
)

print(
    "Best iteration:",
    seed84_best_iteration
)

print(
    "Minimum validation log-loss:",
    round(
        seed84_validation_loss[
            seed84_best_iteration
        ],
        6
    )
)

print(
    "Validation log-loss at last completed iteration:",
    round(
        seed84_validation_loss[-1],
        6
    )
)

# Predictions using the best iteration
seed84_train_probability_best = (
    seed84_best_iteration_model.predict_proba(
        X_train_seed84,
        iteration_range=(
            0,
            seed84_best_iteration + 1
        )
    )[:, 1]
)

seed84_valid_probability_best = (
    seed84_best_iteration_model.predict_proba(
        X_valid_seed84,
        iteration_range=(
            0,
            seed84_best_iteration + 1
        )
    )[:, 1]
)

seed84_train_prediction_best = (
    seed84_train_probability_best
    >= 0.50
).astype(int)

seed84_valid_prediction_best = (
    seed84_valid_probability_best
    >= 0.50
).astype(int)

# Predictions using the last completed iteration
seed84_valid_probability_last = (
    seed84_best_iteration_model.predict_proba(
        X_valid_seed84,
        iteration_range=(
            0,
            seed84_rounds_completed
        )
    )[:, 1]
)

seed84_valid_prediction_last = (
    seed84_valid_probability_last
    >= 0.50
).astype(int)

# Calculate performance
seed84_train_mcc_best = (
    matthews_corrcoef(
        y_train_seed84,
        seed84_train_prediction_best
    )
)

seed84_valid_mcc_best = (
    matthews_corrcoef(
        y_valid_seed84,
        seed84_valid_prediction_best
    )
)

seed84_valid_mcc_last = (
    matthews_corrcoef(
        y_valid_seed84,
        seed84_valid_prediction_last
    )
)

seed84_train_ba_best = (
    balanced_accuracy_score(
        y_train_seed84,
        seed84_train_prediction_best
    )
)

seed84_valid_ba_best = (
    balanced_accuracy_score(
        y_valid_seed84,
        seed84_valid_prediction_best
    )
)

seed84_valid_ba_last = (
    balanced_accuracy_score(
        y_valid_seed84,
        seed84_valid_prediction_last
    )
)

seed84_best_iteration_results = pd.DataFrame([{
    "selected_descriptors":
        len(seed_84_descriptors),

    "rounds_completed":
        seed84_rounds_completed,

    "best_iteration":
        seed84_best_iteration,

    "minimum_validation_logloss":
        seed84_validation_loss[
            seed84_best_iteration
        ],

    "last_validation_logloss":
        seed84_validation_loss[-1],

    "training_mcc_best_iteration":
        seed84_train_mcc_best,

    "validation_mcc_best_iteration":
        seed84_valid_mcc_best,

    "validation_mcc_last_iteration":
        seed84_valid_mcc_last,

    "training_validation_mcc_gap":
        seed84_train_mcc_best
        - seed84_valid_mcc_best,

    "training_balanced_accuracy":
        seed84_train_ba_best,

    "validation_balanced_accuracy_best":
        seed84_valid_ba_best,

    "validation_balanced_accuracy_last":
        seed84_valid_ba_last
}])

display(
    seed84_best_iteration_results
)

seed84_best_iteration_results.to_csv(
    "ga_seed84_best_iteration_results.csv",
    index=False
)

# Plot learning curves
plt.figure(
    figsize=(9, 5)
)

plt.plot(
    range(seed84_rounds_completed),
    seed84_training_loss,
    label="Training log-loss"
)

plt.plot(
    range(seed84_rounds_completed),
    seed84_validation_loss,
    label="Validation log-loss"
)

plt.axvline(
    seed84_best_iteration,
    linestyle="--",
    label=(
        f"Best iteration: "
        f"{seed84_best_iteration}"
    )
)

plt.xlabel("Boosting iteration")
plt.ylabel("Log-loss")

plt.title(
    "GA Seed 84 XGBoost Training "
    "and Validation Log-loss"
)

plt.legend()
plt.tight_layout()

plt.savefig(
    "ga_seed84_best_iteration_learning_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nSaved:")
print(
    "ga_seed84_best_iteration_results.csv"
)

print(
    "ga_seed84_best_iteration_learning_curve.png"
)

In [ ]:
# Refit the seed 84 model and evaluate the test set

test_feature_file = Path(
    "mordred_test_features_filtered.pkl"
)

if not test_feature_file.exists():
    raise FileNotFoundError(
        f"Missing file: {test_feature_file}"
    )

# Load the untouched test feature matrix
X_test_full = pd.read_pickle(
    test_feature_file
).astype("float32")

missing_test_descriptors = [
    descriptor
    for descriptor in seed_84_descriptors
    if descriptor not in X_test_full.columns
]

if missing_test_descriptors:
    raise ValueError(
        "Seed 84 descriptors missing from "
        f"the test data: {missing_test_descriptors[:10]}"
    )

X_test_seed84 = X_test_full[
    seed_84_descriptors
].copy()

# Align the test labels
y_test_seed84 = (
    model_data_seed84.loc[
        X_test_seed84.index,
        "dual_candidate"
    ]
    .astype("int8")
)

if X_test_seed84.isna().any().any():
    raise ValueError(
        "Missing values found in test features."
    )

if y_test_seed84.isna().any():
    raise ValueError(
        "Missing values found in test labels."
    )

# Combine training and validation data
X_train_valid_seed84 = pd.concat(
    [
        X_train_seed84,
        X_valid_seed84
    ],
    axis=0
)

y_train_valid_seed84 = pd.concat(
    [
        y_train_seed84,
        y_valid_seed84
    ],
    axis=0
)

# Calculate balanced weights on the combined data
train_valid_weights_seed84 = (
    compute_sample_weight(
        class_weight="balanced",
        y=y_train_valid_seed84
    )
)

selected_number_of_trees = (
    seed84_best_iteration + 1
)

# Refit using the validation-selected number of trees
seed84_final_model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=selected_number_of_trees,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.10,
    reg_lambda=1.00,
    tree_method="hist",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

seed84_final_model.fit(
    X_train_valid_seed84,
    y_train_valid_seed84,
    sample_weight=train_valid_weights_seed84,
    verbose=False
)

# Generate final test predictions
seed84_test_probability = (
    seed84_final_model.predict_proba(
        X_test_seed84
    )[:, 1]
)

seed84_test_prediction = (
    seed84_test_probability >= 0.50
).astype(int)

# Calculate test metrics
seed84_test_results = pd.DataFrame([{
    "ga_seed": 84,
    "selected_descriptors":
        len(seed_84_descriptors),
    "selected_boosting_iteration":
        seed84_best_iteration,
    "number_of_trees":
        selected_number_of_trees,
    "validation_mcc":
        seed84_valid_mcc_best,
    "validation_balanced_accuracy":
        seed84_valid_ba_best,
    "test_mcc":
        matthews_corrcoef(
            y_test_seed84,
            seed84_test_prediction
        ),
    "test_balanced_accuracy":
        balanced_accuracy_score(
            y_test_seed84,
            seed84_test_prediction
        ),
    "test_precision":
        precision_score(
            y_test_seed84,
            seed84_test_prediction
        ),
    "test_recall":
        recall_score(
            y_test_seed84,
            seed84_test_prediction
        ),
    "test_f1":
        f1_score(
            y_test_seed84,
            seed84_test_prediction
        ),
    "test_roc_auc":
        roc_auc_score(
            y_test_seed84,
            seed84_test_probability
        ),
    "test_average_precision":
        average_precision_score(
            y_test_seed84,
            seed84_test_probability
        )
}])

display(seed84_test_results)

# Save the final results and model
seed84_test_results.to_csv(
    "ga_seed84_final_test_results.csv",
    index=False
)

seed84_final_model.save_model(
    "ga_seed84_final_xgboost_model.json"
)

pd.DataFrame({
    "descriptor": seed_84_descriptors
}).to_csv(
    "ga_seed84_final_descriptors.csv",
    index=False
)

print(
    "\nCombined training and validation shape:",
    X_train_valid_seed84.shape
)

print(
    "Test feature shape:",
    X_test_seed84.shape
)

print(
    "Validation-selected number of trees:",
    selected_number_of_trees
)

print("\nSaved:")
print("ga_seed84_final_test_results.csv")
print("ga_seed84_final_xgboost_model.json")
print("ga_seed84_final_descriptors.csv")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Feature-selection comparison

labels = [
    "Complete Mordred\n(n = 626)",
    "mRMR\n(n = 300)",
    "GA seed 84\n(n = 331)"
]

validation_mcc = np.array([
    0.7034,
    0.7045,
    0.7156
])

test_mcc = np.array([
    0.7087,
    0.7027,
    0.7086
])

x = np.arange(len(labels))

validation_colour = "#B784E8"
test_colour = "#3A8EC1"
connector_colour = "#C9C7E8"
grid_colour = "#D9D9D9"
text_colour = "#1F1F1F"

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10.5,
    "ytick.labelsize": 10.5,
    "legend.fontsize": 10.5
})

fig, ax = plt.subplots(figsize=(8.2, 5.8))

for i in range(len(x)):
    ax.plot(
        [x[i], x[i]],
        [validation_mcc[i], test_mcc[i]],
        color=connector_colour,
        linewidth=2.2,
        zorder=1
    )

# Validation points
ax.scatter(
    x,
    validation_mcc,
    s=95,
    color=validation_colour,
    marker="o",
    label="Validation",
    zorder=3
)

# Scaffold-test points
ax.scatter(
    x,
    test_mcc,
    s=95,
    color=test_colour,
    marker="s",
    label="Scaffold test",
    zorder=3
)

# Exact validation values
for i, value in enumerate(validation_mcc):
    ax.annotate(
        f"{value:.4f}",
        (x[i], value),
        xytext=(8, 7),
        textcoords="offset points",
        ha="left",
        va="bottom",
        fontsize=9,
        color=text_colour
    )

# Exact scaffold-test values
for i, value in enumerate(test_mcc):
    ax.annotate(
        f"{value:.4f}",
        (x[i], value),
        xytext=(8, -8),
        textcoords="offset points",
        ha="left",
        va="top",
        fontsize=9,
        color=text_colour
    )

ax.set_xticks(x)
ax.set_xticklabels(labels)

ax.set_ylabel(
    "Matthews correlation coefficient (MCC)",
    color=text_colour
)

ax.set_xlabel(
    "Descriptor representation",
    color=text_colour
)

ax.set_title(
    "Effect of feature selection on XGBoost performance",
    pad=12,
    color=text_colour
)

ax.set_ylim(0.695, 0.722)

ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.8,
    color=grid_colour,
    alpha=0.7
)

# Clean publication-style axes
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(text_colour)
ax.spines["bottom"].set_color(text_colour)

ax.tick_params(
    axis="both",
    colors=text_colour
)

ax.legend(
    frameon=False,
    loc="upper left"
)

fig.tight_layout()

fig.savefig(
    "figure_3A_feature_selection_mcc.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    "figure_3A_feature_selection_mcc.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# GA stability across random seeds

seeds = np.array([7, 21, 42, 84, 99, 123])

best_validation_mcc = np.array([
    0.7117,
    0.7115,
    0.7129,
    0.7157,
    0.7122,
    0.7114
])

blue = "#3A8EC1"
purple = "#B784E8"
dark_purple = "#7A4FA3"
grid_colour = "#D9D9D9"
text_colour = "#1F1F1F"

mean_mcc = best_validation_mcc.mean()
best_index = np.argmax(best_validation_mcc)

fig, ax = plt.subplots(figsize=(8.2, 5.4))

# All GA runs
ax.scatter(
    seeds,
    best_validation_mcc,
    s=105,
    color=blue,
    edgecolor="white",
    linewidth=1.2,
    zorder=3,
    label="GA runs"
)

# Highlight selected seed 84
ax.scatter(
    seeds[best_index],
    best_validation_mcc[best_index],
    s=190,
    color=purple,
    edgecolor=dark_purple,
    linewidth=1.6,
    zorder=4,
    label="Selected seed 84"
)

# Mean MCC line
ax.axhline(
    mean_mcc,
    color=purple,
    linestyle="--",
    linewidth=1.4,
    alpha=0.85
)

# Mean label
ax.text(
    125,
    mean_mcc + 0.00012,
    f"Mean = {mean_mcc:.4f}",
    ha="right",
    va="bottom",
    fontsize=9.5,
    color=dark_purple
)

# Value labels
for seed, value in zip(seeds, best_validation_mcc):

    offset = 8

    if seed == 84:
        offset = 11

    ax.annotate(
        f"{value:.4f}",
        (seed, value),
        xytext=(0, offset),
        textcoords="offset points",
        ha="center",
        va="bottom",
        fontsize=9,
        color=text_colour
    )

ax.set_xticks(seeds)

ax.set_xlabel(
    "GA random seed",
    fontsize=12,
    color=text_colour
)

ax.set_ylabel(
    "Best validation MCC",
    fontsize=12,
    color=text_colour
)

ax.set_title(
    "GA stability across random seeds",
    fontsize=13,
    pad=12,
    color=text_colour
)

ax.set_ylim(0.7105, 0.7167)

ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.7,
    color=grid_colour,
    alpha=0.7
)

ax.set_axisbelow(True)

# Clean axes
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(text_colour)
ax.spines["bottom"].set_color(text_colour)

ax.tick_params(
    axis="both",
    labelsize=10.5,
    colors=text_colour
)

ax.legend(
    frameon=False,
    loc="upper left",
    fontsize=10
)

fig.tight_layout()

fig.savefig(
    "figure_3B_GA_seed_stability.png",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    "figure_3B_GA_seed_stability.png",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Pairwise Jaccard similarity across GA-selected descriptor sets

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# Locations of the six completed GA runs

seed_folders = {
    7: Path("full_ga_results_seed_7"),
    21: Path("full_ga_results_seed_21"),
    42: Path("full_ga_results"),
    84: Path("full_ga_results_seed_84"),
    99: Path("full_ga_results_seed_99"),
    123: Path("full_ga_results_seed_123")
}

seeds = [7, 21, 42, 84, 99, 123]

# Load selected descriptor names

selected_features = {}

for seed in seeds:

    feature_file = (
        seed_folders[seed]
        / "full_ga_best_features.csv"
    )

    if not feature_file.exists():
        raise FileNotFoundError(
            f"Could not find: {feature_file}"
        )

    df = pd.read_csv(feature_file)

    usable_columns = [
        col for col in df.columns
        if not str(col).lower().startswith("unnamed")
    ]

    if len(usable_columns) == 0:
        raise ValueError(
            f"No descriptor column found in {feature_file}"
        )

    descriptor_column = usable_columns[0]

    selected_features[seed] = set(
        df[descriptor_column]
        .dropna()
        .astype(str)
        .tolist()
    )

    print(
        f"Seed {seed}: "
        f"{len(selected_features[seed])} descriptors"
    )

# Calculate pairwise Jaccard similarity

jaccard_matrix = np.zeros(
    (len(seeds), len(seeds))
)

for i, seed_i in enumerate(seeds):

    for j, seed_j in enumerate(seeds):

        set_i = selected_features[seed_i]
        set_j = selected_features[seed_j]

        intersection = len(
            set_i.intersection(set_j)
        )

        union = len(
            set_i.union(set_j)
        )

        jaccard_matrix[i, j] = (
            intersection / union
        )

# Mean pairwise Jaccard excluding diagonal
upper_triangle = jaccard_matrix[
    np.triu_indices(
        len(seeds),
        k=1
    )
]

mean_jaccard = upper_triangle.mean()

heatmap_cmap = LinearSegmentedColormap.from_list(
    "dissertation_blue_purple",
    [
        "#F4FAFE",
        "#DCC3F5",
        "#8FC3E3",
        "#B784E8",
        "#7A4FA3"
    ]
)

text_colour = "#1F1F1F"

# Plot heatmap

fig, ax = plt.subplots(
    figsize=(7.2, 6.2)
)

image = ax.imshow(
    jaccard_matrix,
    cmap=heatmap_cmap,
    vmin=0,
    vmax=1,
    aspect="equal"
)

# Axis labels
ax.set_xticks(
    np.arange(len(seeds))
)

ax.set_yticks(
    np.arange(len(seeds))
)

ax.set_xticklabels(
    [f"Seed {seed}" for seed in seeds]
)

ax.set_yticklabels(
    [f"Seed {seed}" for seed in seeds]
)

# Values inside cells
for i in range(len(seeds)):

    for j in range(len(seeds)):

        value = jaccard_matrix[i, j]

        text_color = (
            "white"
            if value > 0.60
            else text_colour
        )

        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=9.5,
            color=text_color
        )

# Title
ax.set_title(
    "Similarity of GA-selected descriptor subsets",
    fontsize=13,
    pad=14,
    color=text_colour
)

# Colorbar
colour_bar = fig.colorbar(
    image,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

colour_bar.set_label(
    "Jaccard similarity",
    fontsize=11
)

colour_bar.ax.tick_params(
    labelsize=9
)

# Clean styling
ax.tick_params(
    axis="both",
    labelsize=10,
    length=0
)

for spine in ax.spines.values():
    spine.set_visible(False)

fig.tight_layout()

# Save PNG

figure_file = Path(
    "figure_3C_GA_jaccard_similarity.png"
)

fig.savefig(
    figure_file,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print(
    "\nMean pairwise Jaccard similarity:",
    round(mean_jaccard, 4)
)

print(
    "Saved:",
    figure_file
)

In [ ]:
import pandas as pd

ga_test_pred = pd.read_csv("final_mordred_ga_test_predictions.csv")

print(ga_test_pred.columns.tolist())
print()
display(ga_test_pred.head())

In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, balanced_accuracy_score, matthews_corrcoef

df = pd.read_csv("final_mordred_ga_test_predictions.csv")

y_true = df["true_label"]
y_pred = df["predicted_label"]

print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
print("MCC:", matthews_corrcoef(y_true, y_pred))
print("Rows:", len(df))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix
from matplotlib.colors import LinearSegmentedColormap, PowerNorm


df = pd.read_csv(
    "ga_seed84_final_test_predictions_recreated.csv"
)

y_true = df["true_label"].astype(int)
y_pred = df["predicted_label"].astype(int)

cm = confusion_matrix(y_true, y_pred)

print("Confusion matrix:")
print(cm)


light_blue   = "#8FC3E3"
blue         = "#3A8EC1"
dark_blue    = "#246A94"

light_purple = "#DCC3F5"
purple       = "#B784E8"
dark_purple  = "#7A4FA3"

text_colour  = "#1F1F1F"

thesis_cmap = LinearSegmentedColormap.from_list(
    "tanya_blue_purple",
    [
        "#F7F9FC",
        light_blue,
        blue,
        purple,
        dark_purple
    ]
)

fig, ax = plt.subplots(
    figsize=(6.6, 5.6),
    facecolor="white"
)

ax.set_facecolor("white")

im = ax.imshow(
    cm,
    cmap=thesis_cmap,
    norm=PowerNorm(
        gamma=0.55,
        vmin=0,
        vmax=cm.max()
    )
)

class_labels = [
    "Non-dual candidate",
    "Dual candidate"
]

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])

ax.set_xticklabels(
    class_labels,
    fontsize=10.5,
    color=text_colour
)

ax.set_yticklabels(
    class_labels,
    fontsize=10.5,
    color=text_colour
)

ax.set_xlabel(
    "Predicted class",
    fontsize=12,
    color=text_colour,
    labelpad=10
)

ax.set_ylabel(
    "True class",
    fontsize=12,
    color=text_colour,
    labelpad=10
)

ax.set_title(
    "Final GA-selected XGBoost classifier",
    fontsize=14,
    color=text_colour,
    pad=14
)

for i in range(2):
    for j in range(2):

        value = cm[i, j]

        normalized_value = (
            PowerNorm(
                gamma=0.55,
                vmin=0,
                vmax=cm.max()
            )(value)
        )

        annotation_colour = (
            "white"
            if normalized_value > 0.60
            else text_colour
        )

        ax.text(
            j,
            i,
            f"{value:,}",
            ha="center",
            va="center",
            fontsize=15,
            fontweight="semibold",
            color=annotation_colour
        )
        
ax.set_xticks(
    np.arange(-0.5, 2, 1),
    minor=True
)

ax.set_yticks(
    np.arange(-0.5, 2, 1),
    minor=True
)

ax.grid(
    which="minor",
    color="white",
    linewidth=2.5
)

ax.tick_params(
    which="minor",
    bottom=False,
    left=False
)

ax.tick_params(
    axis="both",
    length=0
)

for spine in ax.spines.values():
    spine.set_visible(False)

cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.045,
    pad=0.045
)

cbar.set_label(
    "Number of molecules",
    fontsize=10.5,
    color=text_colour,
    labelpad=9
)

cbar.ax.tick_params(
    labelsize=9,
    colors=text_colour
)

cbar.outline.set_linewidth(0.6)
cbar.outline.set_edgecolor("#B5B5B5")

fig.text(
    0.5,
    0.035,
    "Scaffold test set  •  decision threshold = 0.50  •  n = 14,700",
    ha="center",
    fontsize=9.2,
    color="#666666"
)

fig.subplots_adjust(
    left=0.18,
    right=0.90,
    bottom=0.16,
    top=0.88
)

figure_file = "figure_final_seed84_confusion_matrix.png"

fig.savefig(
    figure_file,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Saved:", figure_file)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

models = [
    "HGB direct",
    "XGBoost direct",
    "XGBoost direct",
    "Regression-derived",
    "Soft voting"
]

balanced_accuracy = [0.863, 0.876, 0.860, 0.788, 0.854]
mcc = [0.662, 0.660, 0.687, 0.654, 0.695]

annotations = [
    "Threshold = 0.50",
    "Threshold = 0.70",
    "Threshold = 0.50",
    "Score rule ≤ −8.0",
    "Threshold = 0.40"
]

blue = "#3A8EC1"
purple = "#B784E8"

dark_blue = "#246A94"
dark_purple = "#7A4FA3"

grid_colour = "#D9D9D9"
text_colour = "#1F1F1F"

x = np.arange(len(models))
width = 0.34

fig, ax = plt.subplots(figsize=(10.8, 6.2), facecolor="white")

bars1 = ax.bar(
    x - width/2,
    balanced_accuracy,
    width,
    label="Balanced accuracy",
    color=blue,
    edgecolor=dark_blue,
    linewidth=0.8
)

bars2 = ax.bar(
    x + width/2,
    mcc,
    width,
    label="MCC",
    color=purple,
    edgecolor=dark_purple,
    linewidth=0.8
)

for bar in bars1:
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.006,
        f"{bar.get_height():.3f}",
        ha="center",
        va="bottom",
        fontsize=9.5,
        color=text_colour
    )

for bar in bars2:
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.006,
        f"{bar.get_height():.3f}",
        ha="center",
        va="bottom",
        fontsize=9.5,
        color=text_colour
    )

for i, label in enumerate(annotations):
    ax.text(
        x[i],
        0.907,
        label,
        ha="center",
        va="center",
        fontsize=8.7,
        color="#555555"
    )

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=10)

ax.set_ylabel(
    "Validation performance",
    fontsize=11
)

ax.set_xlabel(
    "Modelling approach",
    fontsize=11,
    labelpad=8
)

ax.set_ylim(0.60, 0.925)

ax.yaxis.grid(
    linestyle="--",
    linewidth=0.6,
    color=grid_colour,
    alpha=0.75
)

ax.set_axisbelow(True)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.tick_params(axis="y", labelsize=9.5)

fig.suptitle(
    "Comparison of baseline dual-candidate classification approaches",
    fontsize=14,
    color=text_colour,
    y=0.965
)

ax.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.16),
    ncol=2,
    fontsize=10
)

fig.subplots_adjust(
    top=0.78,
    bottom=0.15,
    left=0.10,
    right=0.98
)

figure_file = "figure6_baseline_classification_final.png"

fig.savefig(
    figure_file,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Saved:", figure_file)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import matplotlib.patheffects as pe

layers = [
    ("Input",              "100 MI-ranked descriptors",     "io"),
    ("Linear | out=2000",  "(1, 100) → (1, 2000)",           "linear"),
    ("ReLU",                "(1, 2000) → (1, 2000)",         "activation"),
    ("Linear | out=2",      "(1, 2000) → (1, 2)",             "linear"),
    ("Output",              "JNK3 & GSK3β docking scores",   "io"),
]

colours = {
    "linear":     ("#DCEBF9", "#4E86C1", "#1B3A5C"),  
    "activation": ("#EAE0F5", "#9B6FC9", "#3E2657"),  
    "io":         ("#F2F2F2", "#8C8C8C", "#2C2C2C"),  
}

TITLE_FS, SUB_FS = 16, 13

box_h = 1.15
gap = 0.85
min_box_w = 3.6           
pad_x = 0.9                
n = len(layers)
fig_w = 7.6

fig, ax = plt.subplots(figsize=(fig_w, n * box_h + (n - 1) * gap + 1.2))
fig_h = fig.get_size_inches()[1]
ax.set_xlim(0, fig_w)
ax.set_ylim(0, fig_h)
ax.axis("off")

renderer = fig.canvas.get_renderer()

def text_width_in(txt, fontsize, weight="normal"):
    t = ax.text(0, 0, txt, fontsize=fontsize, fontweight=weight,
                 family="DejaVu Sans")
    bbox = t.get_window_extent(renderer=renderer)
    t.remove()
    return bbox.width / fig.dpi

y = fig_h - 0.6
centers = []

for title, subtitle, key in layers:
    fill, edge, textcol = colours[key]

    w_title = text_width_in(title, TITLE_FS, "bold")
    w_sub = text_width_in(subtitle, SUB_FS)
    box_w = max(min_box_w, w_title + pad_x, w_sub + pad_x)

    x = (fig_w - box_w) / 2
    y_top = y
    y_bottom = y - box_h

    # subtle drop shadow
    shadow = FancyBboxPatch(
        (x + 0.045, y_bottom - 0.045), box_w, box_h,
        boxstyle="round,pad=0.02,rounding_size=0.16",
        linewidth=0, facecolor="#000000", alpha=0.10, zorder=1,
    )
    ax.add_patch(shadow)

    box = FancyBboxPatch(
        (x, y_bottom), box_w, box_h,
        boxstyle="round,pad=0.02,rounding_size=0.16",
        linewidth=1.6, edgecolor=edge, facecolor=fill, zorder=2,
    )
    ax.add_patch(box)

    cx, cy = x + box_w / 2, y_bottom + box_h / 2
    ax.text(cx, cy + 0.22, title, ha="center", va="center",
             fontsize=TITLE_FS, fontweight="bold", color=textcol,
             family="DejaVu Sans", zorder=3)
    ax.text(cx, cy - 0.24, subtitle, ha="center", va="center",
             fontsize=SUB_FS, color=textcol, family="DejaVu Sans", zorder=3)

    centers.append((cx, y_top, y_bottom))
    y = y_bottom - gap

axis_x = fig_w / 2
for i in range(len(centers) - 1):
    _, _, bottom_i = centers[i]
    _, top_ip1, _ = centers[i + 1]
    arrow = FancyArrowPatch(
        (axis_x, bottom_i), (axis_x, top_ip1),
        arrowstyle="-|>", mutation_scale=22,
        linewidth=1.8, color="#4D4D4D", zorder=1.5,
        path_effects=[pe.Stroke(linewidth=1.8, foreground="#4D4D4D"), pe.Normal()],
    )
    ax.add_patch(arrow)

plt.tight_layout()

out_path = "nn_architecture_flowchart.png"
fig.savefig(out_path, dpi=600, bbox_inches="tight", facecolor="white")
print("Saved:", out_path)

In [ ]:
%pip install shap

In [ ]:
import shap
print(shap.__version__)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import xgboost as xgb

# Files from the final seed-84 model

descriptor_file = "ga_seed84_final_descriptors.csv"
test_features_file = "mordred_test_features_filtered.pkl"
model_file = "ga_seed84_final_xgboost_model.json"

# Load final 331 selected descriptors

descriptor_df = pd.read_csv(descriptor_file)

print("Descriptor file columns:", descriptor_df.columns.tolist())

# Use the first column containing the descriptor names
selected_features = descriptor_df.iloc[:, 0].astype(str).tolist()

print("Number of selected descriptors:", len(selected_features))
print("First 10 descriptors:")
print(selected_features[:10])

# Load scaffold-test features

X_test_full = pd.read_pickle(test_features_file)

X_test_ga = X_test_full[selected_features].copy()

print("\nFull test feature matrix:", X_test_full.shape)
print("GA-selected test matrix:", X_test_ga.shape)

# Load final XGBoost model

model = xgb.XGBClassifier()
model.load_model(model_file)

print("\nFinal XGBoost model loaded successfully.")

In [ ]:
# Reproducible sample for post-hoc SHAP interpretation

SHAP_SAMPLE_SIZE = 2000
SHAP_RANDOM_SEED = 42

X_shap = X_test_ga.sample(
    n=min(SHAP_SAMPLE_SIZE, len(X_test_ga)),
    random_state=SHAP_RANDOM_SEED
).copy()

print("SHAP sample shape:", X_shap.shape)

In [ ]:
# TreeSHAP

explainer = shap.TreeExplainer(model)

shap_values = explainer(X_shap)

print("SHAP values calculated.")
print("SHAP value array shape:", shap_values.values.shape)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

thesis_cmap = LinearSegmentedColormap.from_list(
    "thesis_blue_purple",
    [
        "#8FC3E3",   # light blue
        "#3A8EC1",   # blue
        "#B784E8",   # purple
        "#7A4FA3"    # dark purple
    ]
)

fig, ax = plt.subplots(figsize=(9.2, 6.3))

shap.plots.beeswarm(
    shap_values,
    max_display=10,
    group_remaining_features=False,
    color=thesis_cmap,
    show=False,
    plot_size=None,
    ax=ax
)

ax.set_xlabel(
    "SHAP value (impact on model output)",
    fontsize=11
)

ax.set_title(
    "Global feature attribution for the final GA-selected XGBoost classifier",
    fontsize=13,
    pad=14
)

ax.tick_params(
    axis="both",
    labelsize=10
)

plt.tight_layout()

figure_file = "figure_final_xgboost_shap_beeswarm_top10_thesis_palette.png"

plt.savefig(
    figure_file,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Saved:", figure_file)

In [ ]:
# Global mean absolute SHAP importance

mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

shap_importance = pd.DataFrame({
    "Descriptor": X_shap.columns,
    "Mean_abs_SHAP": mean_abs_shap
}).sort_values(
    "Mean_abs_SHAP",
    ascending=False
).reset_index(drop=True)

print("Top 10 descriptors:")
display(shap_importance.head(10))

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Load the final seed-84 model
seed84_model = xgb.XGBClassifier()
seed84_model.load_model("ga_seed84_final_xgboost_model.json")

# Load validation features and restrict to the 331 seed-84 descriptors 
X_validation_all = pd.read_pickle("mordred_validation_features_filtered.pkl")

seed84_features = pd.read_csv(
    "full_ga_results_seed_84/full_ga_best_features.csv"
)
usable_columns = [c for c in seed84_features.columns if not str(c).lower().startswith("unnamed")]
seed84_feature_list = seed84_features[usable_columns[0]].dropna().astype(str).tolist()

X_validation_seed84 = X_validation_all[seed84_feature_list]

# Load validation labels 
data_df = pd.read_csv("preprocessed_model_data.csv")
validation_df = data_df[data_df["split"] == "validation"].copy()

y_validation = validation_df.loc[X_validation_seed84.index, "dual_candidate"].astype(int)

print("Validation shape:", X_validation_seed84.shape)
print("Validation labels:", y_validation.shape)

# Get predicted probabilities 
validation_probabilities = seed84_model.predict_proba(X_validation_seed84)[:, 1]

# Sweep thresholds
threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.01):

    predicted_labels = (validation_probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_validation, predicted_labels).ravel()
    specificity = tn / (tn + fp)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "balanced_accuracy": balanced_accuracy_score(y_validation, predicted_labels),
        "mcc": matthews_corrcoef(y_validation, predicted_labels),
        "precision": precision_score(y_validation, predicted_labels, zero_division=0),
        "recall": recall_score(y_validation, predicted_labels, zero_division=0),
        "specificity": specificity,
        "f1_score": f1_score(y_validation, predicted_labels, zero_division=0),
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df.to_csv(
    "ga_seed84_final_model_threshold_sweep.csv",
    index=False
)

print("\nSaved: ga_seed84_final_model_threshold_sweep.csv")
print("\nBest balanced accuracy:")
print(threshold_results_df.sort_values("balanced_accuracy", ascending=False).head(3))

print("\nBest MCC:")
print(threshold_results_df.sort_values("mcc", ascending=False).head(3))

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb

from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    balanced_accuracy_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# load the mordred features for train and validation
X_train_all = pd.read_pickle("mordred_training_features_filtered.pkl")
X_validation_all = pd.read_pickle("mordred_validation_features_filtered.pkl")

# load the seed 84 selected descriptor list
seed84_features = pd.read_csv("full_ga_results_seed_84/full_ga_best_features.csv")
usable_columns = [c for c in seed84_features.columns if not str(c).lower().startswith("unnamed")]
seed84_feature_list = seed84_features[usable_columns[0]].dropna().astype(str).tolist()

X_train_seed84 = X_train_all[seed84_feature_list]
X_validation_seed84 = X_validation_all[seed84_feature_list]

# load labels
data_df = pd.read_csv("preprocessed_model_data.csv")
train_df = data_df[data_df["split"] == "train"].copy()
validation_df = data_df[data_df["split"] == "validation"].copy()

y_train = train_df.loc[X_train_seed84.index, "dual_candidate"].astype(int)
y_validation = validation_df.loc[X_validation_seed84.index, "dual_candidate"].astype(int)

print("Training shape:", X_train_seed84.shape)
print("Validation shape:", X_validation_seed84.shape)

# balanced sample weights for training
train_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

# same settings used originally for the seed 84 best iteration model
seed84_model = XGBClassifier(
    objective="binary:logistic",
    n_estimators=3000,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.80,
    colsample_bytree=0.80,
    reg_alpha=0.10,
    reg_lambda=1.00,
    tree_method="hist",
    eval_metric="logloss",
    early_stopping_rounds=200,
    random_state=42,
    n_jobs=-1
)

# train only on the training set, validation is only used to pick the best iteration
seed84_model.fit(
    X_train_seed84,
    y_train,
    sample_weight=train_weights,
    eval_set=[
        (X_train_seed84, y_train),
        (X_validation_seed84, y_validation)
    ],
    verbose=False
)

best_iteration = seed84_model.best_iteration
print("Best iteration:", best_iteration)

# predict validation probabilities using the best iteration only
validation_probabilities = seed84_model.predict_proba(
    X_validation_seed84,
    iteration_range=(0, best_iteration + 1)
)[:, 1]

# sweep thresholds
threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.01):

    predicted_labels = (validation_probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_validation, predicted_labels).ravel()
    specificity = tn / (tn + fp)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "balanced_accuracy": balanced_accuracy_score(y_validation, predicted_labels),
        "mcc": matthews_corrcoef(y_validation, predicted_labels),
        "precision": precision_score(y_validation, predicted_labels, zero_division=0),
        "recall": recall_score(y_validation, predicted_labels, zero_division=0),
        "specificity": specificity,
        "f1_score": f1_score(y_validation, predicted_labels, zero_division=0),
        "true_negative": tn,
        "false_positive": fp,
        "false_negative": fn,
        "true_positive": tp
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df.to_csv(
    "ga_seed84_final_model_threshold_sweep_correct.csv",
    index=False
)

print("Saved: ga_seed84_final_model_threshold_sweep_correct.csv")

print("\nBest balanced accuracy:")
print(threshold_results_df.sort_values("balanced_accuracy", ascending=False).head(3))

print("\nBest MCC:")
print(threshold_results_df.sort_values("mcc", ascending=False).head(3))

In [ ]:
import pandas as pd
import xgboost as xgb

# load the final seed 84 model
seed84_model = xgb.XGBClassifier()
seed84_model.load_model("ga_seed84_final_xgboost_model.json")

# load the mordred test features
X_test_all = pd.read_pickle("mordred_test_features_filtered.pkl")

# load the seed 84 selected descriptor list
seed84_features = pd.read_csv("full_ga_results_seed_84/full_ga_best_features.csv")
usable_columns = [c for c in seed84_features.columns if not str(c).lower().startswith("unnamed")]
seed84_feature_list = seed84_features[usable_columns[0]].dropna().astype(str).tolist()

# keep only the 331 selected descriptors
X_test_seed84 = X_test_all[seed84_feature_list]

# load the test labels and smiles
data_df = pd.read_csv("preprocessed_model_data.csv")
test_df = data_df[data_df["split"] == "test"].copy()

y_test = test_df.loc[X_test_seed84.index, "dual_candidate"].astype(int)
smiles_test = test_df.loc[X_test_seed84.index, "canonical_smiles"]

# get predictions from the final model
test_probability = seed84_model.predict_proba(X_test_seed84)[:, 1]
test_prediction = (test_probability >= 0.50).astype(int)

# put everything together in one table
results_df = pd.DataFrame({
    "canonical_smiles": smiles_test.values,
    "true_label": y_test.values,
    "predicted_probability": test_probability,
    "predicted_label": test_prediction
})

results_df.to_csv(
    "ga_seed84_final_test_predictions_with_smiles.csv",
    index=False
)

print("Saved: ga_seed84_final_test_predictions_with_smiles.csv")
print("Rows:", len(results_df))

In [ ]:
import pandas as pd

# load the mordred test features
X_test_all = pd.read_pickle("mordred_test_features_filtered.pkl")

# the top 10 descriptors from the SHAP analysis 
top_shap_descriptors = [
    "fragCpx",
    "RotRatio",
    "n6aRing",
    "TpiPC10",
    "C3SP2",
    "GATS2are",
    "BCUTs-1l",
    "BCUTare-1l",
    "nRing",
    "AMID"
]

# check all of them are actually in the test data
missing = [d for d in top_shap_descriptors if d not in X_test_all.columns]
print("Missing descriptors:", missing)

available_descriptors = [d for d in top_shap_descriptors if d in X_test_all.columns]

X_test_top = X_test_all[available_descriptors].copy()

# load test labels and smiles so results can be matched up later
data_df = pd.read_csv("preprocessed_model_data.csv")
test_df = data_df[data_df["split"] == "test"].copy()

X_test_top["canonical_smiles"] = test_df.loc[X_test_top.index, "canonical_smiles"]

X_test_top.to_csv(
    "top_shap_descriptor_values_test.csv",
    index=False
)

print("Saved: top_shap_descriptor_values_test.csv")
print("Rows:", len(X_test_top))
print("Columns:", X_test_top.columns.tolist())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Fragments, Descriptors

blue = "#3A8EC1"
dark_purple = "#7A4FA3"
grid_colour = "#D9D9D9"
text_colour = "#1F1F1F"

plt.rcParams['font.size'] = 10.5

# load the final model predictions with smiles
df = pd.read_csv("ga_seed84_final_test_predictions_with_smiles.csv")
df["correct"] = df["true_label"] == df["predicted_label"]

# classify each molecule into a chemical family based on functional groups
def classify_family(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "Other"
    if Fragments.fr_ester(mol) > 0:
        return "Esters"
    if Fragments.fr_COO(mol) > 0 or Fragments.fr_COO2(mol) > 0:
        return "Carboxylic acids"
    has_amide = Fragments.fr_amide(mol) > 0
    if not has_amide:
        return "Non-amide compounds"
    if Fragments.fr_halogen(mol) > 0:
        return "Halogenated amides"
    if Fragments.fr_ether(mol) > 0:
        return "Ether-containing amides"
    if Fragments.fr_NH1(mol) > 0:
        return "Amides with free amine (NH)"
    return "Other amides"

df["chemical_family"] = df["canonical_smiles"].apply(classify_family)

family_summary = df.groupby("chemical_family")["correct"].agg(["mean", "count"])
family_summary = family_summary.sort_values("mean")

print(family_summary)

# amide-based variants vs distinct functional groups
amide_variant_families = [
    "Amides with free amine (NH)",
    "Halogenated amides",
    "Ether-containing amides",
    "Other amides"
]

family_colours = [
    dark_purple if name in amide_variant_families else blue
    for name in family_summary.index
]

fig, ax = plt.subplots(figsize=(9.5, 7))
bars = ax.barh(family_summary.index, family_summary["mean"], color=family_colours, alpha=0.85, height=0.6)

for bar, mean, count in zip(bars, family_summary["mean"], family_summary["count"]):
    ax.annotate(f"{mean:.3f}  (n={count})",
                xy=(mean, bar.get_y() + bar.get_height()/2),
                xytext=(6, 0), textcoords="offset points",
                va="center", fontsize=9.5, fontweight="bold", color=text_colour)

ax.set_xlabel("Classification accuracy", fontsize=12, fontweight="bold", color=text_colour)
ax.set_xlim(0, 1.15)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(grid_colour)
ax.spines["bottom"].set_color(grid_colour)
ax.tick_params(colors=text_colour, labelsize=10.5)
ax.grid(axis="x", color=grid_colour, linewidth=0.7, alpha=0.7)
ax.set_axisbelow(True)

legend_elements = [
    plt.Rectangle((0,0),1,1, color=dark_purple, alpha=0.85, label="Amide-based variants"),
    plt.Rectangle((0,0),1,1, color=blue, alpha=0.85, label="Distinct functional groups")
]
ax.legend(handles=legend_elements, loc="upper center", frameon=False, fontsize=10,
          labelcolor=text_colour, ncol=2, bbox_to_anchor=(0.5, 1.1))

plt.tight_layout()
plt.savefig("figure_accuracy_by_chemical_family.png", dpi=600, bbox_inches="tight")
plt.show()

print("Saved: figure_accuracy_by_chemical_family.png")

# compute molecular weight and split into linear bins
df["mol_weight"] = df["canonical_smiles"].apply(
    lambda smi: Descriptors.MolWt(Chem.MolFromSmiles(smi))
)

n_groups = 6
min_mw = df["mol_weight"].min()
max_mw = df["mol_weight"].max()
bin_edges = np.linspace(min_mw, max_mw, n_groups + 1)

df["mw_group"] = pd.cut(df["mol_weight"], bins=bin_edges, include_lowest=True)

mw_summary = df.groupby("mw_group", observed=True)["correct"].agg(["mean", "count"])
print(mw_summary)

labels = [f"{bin_edges[i]:.0f}-{bin_edges[i+1]:.0f}" for i in range(n_groups)]

reliable_threshold = 100
mw_colours = [blue if c >= reliable_threshold else dark_purple for c in mw_summary["count"]]

fig, ax = plt.subplots(figsize=(8.8, 6.2))
bars = ax.bar(labels, mw_summary["mean"], color=mw_colours, alpha=0.85, width=0.65)

for bar, mean, count in zip(bars, mw_summary["mean"], mw_summary["count"]):
    ax.annotate(f"{mean:.3f}\n(n={count})",
                xy=(bar.get_x() + bar.get_width()/2, mean),
                xytext=(0, 6), textcoords="offset points",
                ha="center", fontsize=9, fontweight="bold", color=text_colour)

ax.set_xlabel("Molecular weight range (linear bins)", fontsize=12, fontweight="bold", color=text_colour)
ax.set_ylabel("Classification accuracy", fontsize=12, fontweight="bold", color=text_colour)
ax.set_ylim(0, 1.18)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(grid_colour)
ax.spines["bottom"].set_color(grid_colour)
ax.tick_params(colors=text_colour, labelsize=10)
ax.grid(axis="y", color=grid_colour, linewidth=0.7, alpha=0.7)
ax.set_axisbelow(True)

legend_elements = [
    plt.Rectangle((0,0),1,1, color=blue, alpha=0.85, label=f"Adequately sampled (n>={reliable_threshold})"),
    plt.Rectangle((0,0),1,1, color=dark_purple, alpha=0.85, label=f"Small sample (n<{reliable_threshold})")
]
ax.legend(handles=legend_elements, loc="upper center", frameon=False, fontsize=9.5,
          labelcolor=text_colour, ncol=2, bbox_to_anchor=(0.5, 1.13))

plt.tight_layout()
plt.savefig("figure_accuracy_by_mw_linear.png", dpi=600, bbox_inches="tight")
plt.show()

print("Saved: figure_accuracy_by_mw_linear.png")

df.to_csv("test_predictions_with_family_and_mw.csv", index=False)
print("Saved: test_predictions_with_family_and_mw.csv")